<a href="https://colab.research.google.com/github/joppetk/oil_spill_detection/blob/main/notebooks/DeepLabV3_OilSpill_Training_for_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeepLabV3 (ResNet‑50) — Oil Spill Segmentation (PyTorch, Colab)
This notebook trains a **DeepLabV3** model on your Sentinel‑1 oil-spill dataset stored in Google Drive.  
Assumptions:
- Inputs are **2‑channel** GeoTIFFs (VV, VH) already converted to **dB**.
- Masks are **binary** (0=background, 1=oil), same H×W as inputs.
- Dataset is arranged like:
```
/MyDrive/datasets/oil_spill/
  images/   # *.tif  (2 channels: VV, VH)
  masks/    # *.png or *.tif (single-channel 0/1)
```
You can adjust paths as needed.


In [ ]:
!pip -q install scikit-learn
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install rasterio albumentations==1.4.7 opencv-python matplotlib tqdm lightning==2.3.3

In [ ]:
!python --version

In [ ]:
#@title ⛏️ Setup


import os, sys, math, time, json, random, pathlib
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision.models.segmentation import deeplabv3_resnet101
from torchvision.transforms.functional import normalize as tv_normalize
import rasterio
import cv2
import csv
from datetime import datetime

from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.amp import GradScaler, autocast

print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())

In [ ]:
#@title 🔗 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Cfg:
    drive_root: str = '/content/drive/MyDrive'
    data_dir: str = '/content/drive/MyDrive/Sentinel SAR datasets'

    # --- TRAINING SET FOLDERS ---
    oil_img_dir: str       = 'Oil'
    oil_mask_dir: str      = 'Mask_oil'
    no_oil_img_dir: str    = 'No_oil'
    no_oil_mask_dir: str   = 'Mask_no_oil'
    look_img_dir: str      = 'Lookalike'
    look_mask_dir: str     = 'Mask_lookalike'

    # --- HOLDOUT TEST SET (optional, not used in K-fold yet) ---
    test_img_dir: str  = '02_Test_images_and_ground_truth/Images'
    test_mask_dir: str = '02_Test_images_and_ground_truth/Mask'

     # --- HOLDOUT TEST SET (optional, not used in K-fold yet) ---
    test_oil_img_dir: str  = '02_Test_images_and_ground_truth/Images/Oil'
    test_oil_mask_dir: str = '02_Test_images_and_ground_truth/Mask/Oil'
    test_no_oil_img_dir: str  = '02_Test_images_and_ground_truth/Images/No oil'
    test_no_oil_mask_dir: str = '02_Test_images_and_ground_truth/Mask/No oil'
    test_look_img_dir: str  = '02_Test_images_and_ground_truth/Images/Lookalike'
    test_look_mask_dir: str = '02_Test_images_and_ground_truth/Mask/Lookalike'

    out_dir: str = '/content/drive/MyDrive/oil_spill_checkpoints/periments/deepLabV3'






    num_classes: int = 2  # background, oil
    epochs: int = 50
    batch_size: int = 8
    lr: float = 5e-4
    weight_decay: float = 1e-4
    crop_size: int = 512
    amp: bool = True
    seed: int = 1337

    # --- K-FOLD / CALIBRATION SETTINGS ---
    k_folds: int = 5
    n_bins: int = 10      # reliability bins for ECE


    log_file: str = field(init=False)

    # This method runs *after* the instance (cfg) is created
    def __post_init__(self):
        # 1. Define the full path for the log file
        #self.log_file = os.path.join(self.out_dir, "deepLab_resnet101_training_log.csv")
        #new experiment using resnet101 for oil only dataset
        self.log_file = os.path.join(self.out_dir, "deepLab_resnet101_training_test_data_included_in_log2.csv")

        # 2. Ensure the output directory exists
        os.makedirs(self.out_dir, exist_ok=True)

        # 3. Write headers only if the file doesn't exist
        if not os.path.isfile(self.log_file):
            print(f"Creating new log file: {self.log_file}")
            with open(self.log_file, mode="w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([
                    "timestamp", "fold", "epoch", "train_loss",
                    "val_loss", "IoU", "F1", "FAR", "ECE", "uncertainty_var", "test_iou", "test_f1"
                ])

cfg = Cfg()
os.makedirs(cfg.out_dir, exist_ok=True)


In [ ]:
from typing import List, Dict

def build_training_items(cfg: Cfg) -> List[Dict]:
    """Collect all (image, mask) pairs from Oil, No_oil, Lookalike."""
    items = []

    def add_pairs(img_subdir, mask_subdir, label):
        img_dir = os.path.join(cfg.data_dir, img_subdir)
        mask_dir = os.path.join(cfg.data_dir, mask_subdir)
        assert os.path.isdir(img_dir), f"Missing folder: {img_dir}"
        assert os.path.isdir(mask_dir), f"Missing folder: {mask_dir}"

        for name in sorted(os.listdir(img_dir)):
            if not name.lower().endswith('.tif'):
                continue
            img_path = os.path.join(img_dir, name)
            base = os.path.splitext(name)[0]
            if img_subdir in (cfg.test_oil_img_dir, cfg.test_no_oil_img_dir, cfg.test_look_img_dir):
              base = base + '_segmentation'
            # mask can be .png or .tif
            mask_png = os.path.join(mask_dir, base + '.png')
            mask_tif = os.path.join(mask_dir, base + '.tif')
            if os.path.exists(mask_png):
                mask_path = mask_png
            elif os.path.exists(mask_tif):
                mask_path = mask_tif
            else:
                print(f"[WARN] No mask found for {img_path}")
                continue

            items.append({
                'img': img_path,
                'mask': mask_path,
                'subset': label,
                'name': name,
            })

    add_pairs(cfg.oil_img_dir,     cfg.oil_mask_dir,     'oil')
    add_pairs(cfg.no_oil_img_dir,  cfg.no_oil_mask_dir,  'no_oil')
    add_pairs(cfg.look_img_dir,    cfg.look_mask_dir,    'lookalike')

    add_pairs(cfg.test_oil_img_dir,   cfg.test_oil_mask_dir,    'test_oil')
    add_pairs(cfg.test_no_oil_img_dir,   cfg.test_no_oil_mask_dir,    'test_no_oil')
    add_pairs(cfg.test_look_img_dir,   cfg.test_look_mask_dir,    'test_look')

    print(f"Total training items: {len(items)} "
          f"(oil={sum(i['subset']=='oil' for i in items)}, "
          f"no_oil={sum(i['subset']=='no_oil' for i in items)}, "
          f"lookalike={sum(i['subset']=='lookalike' for i in items)})")
    return items

all_items = build_training_items(cfg)


In [ ]:
from torch.utils.data import Sampler
import random

class BalancedOilSampler(Sampler):
    """
    Sampler that oversamples/undersamples by subset label to approximate
    a desired per-sample subset distribution, e.g.:
      oil: 0.5, no_oil: 0.25, lookalike: 0.25

    items: the same list of dicts you pass to OilSpillDataset
           each item must have a 'subset' key.
    """
    def __init__(self, items, target_probs=None, num_samples=None, seed=1337):
        """
        target_probs: dict like {'oil':0.5, 'no_oil':0.25, 'lookalike':0.25}
        num_samples : how many samples per epoch (default = len(items))
        """
        self.items = items
        self.rng = random.Random(seed)

        # default ratios ~50/25/25
        if target_probs is None:
            target_probs = {'oil': 0.5, 'no_oil': 0.25, 'lookalike': 0.25}

        self.labels = ['oil', 'no_oil', 'lookalike']
        self.probs = [target_probs.get(l, 0.0) for l in self.labels]
        s = sum(self.probs)
        if s <= 0:
            raise ValueError("target_probs must have positive sum")
        self.probs = [p / s for p in self.probs]

        # collect indices per subset
        self.indices_by_subset = {label: [] for label in self.labels}
        for idx, item in enumerate(items):
            subset = item.get('subset')
            if subset in self.indices_by_subset:
                self.indices_by_subset[subset].append(idx)

        # sanity: each subset used in probs must have indices
        for label, p in zip(self.labels, self.probs):
            if p > 0 and len(self.indices_by_subset[label]) == 0:
                raise ValueError(f"No items for subset '{label}' but prob>0")

        # how many samples in one epoch
        self.num_samples = num_samples if num_samples is not None else len(items)

    def __iter__(self):
        # yield indices according to subset probabilities
        for _ in range(self.num_samples):
            subset = self.rng.choices(self.labels, weights=self.probs, k=1)[0]
            idx_list = self.indices_by_subset[subset]
            yield self.rng.choice(idx_list)

    def __len__(self):
        return self.num_samples


In [ ]:
test_oil_items = [i for i in all_items if i['subset'] == 'test_oil']
test_no_oil_items = [i for i in all_items if i['subset'] == 'test_no_oil']
test_look_items = [i for i in all_items if i['subset'] == 'test_look']
print(test_oil_items[0])

This is old - handles only VV+VH

In [ ]:
class OilSpillDataset(Dataset):
    def __init__(self, items, crop_size=512, augment=True):
        """
        items: list of dicts with keys ['img', 'mask', 'subset', 'name']
        """
        self.items = items
        self.crop = crop_size
        self.augment = augment

    def __len__(self):
        return len(self.items)

    def _read_tiff_2ch(self, path):
        with rasterio.open(path) as ds:
            arr = ds.read()  # (C,H,W) or (H,W)
        # Force to 2 bands [VV, VH]
        if arr.ndim == 3 and arr.shape[0] >= 2:
            arr = arr[:2]
        elif arr.ndim == 3 and arr.shape[0] == 1:
            arr = np.concatenate([arr, arr], 0)
        elif arr.ndim == 2:
            arr = np.stack([arr, arr], 0)
        else:
            raise ValueError(f"Unexpected TIFF shape {arr.shape} for {path}")
        arr = np.transpose(arr, (1,2,0)).astype(np.float32)  # (H,W,2)
        return arr

    def _read_mask(self, path):
        if path.lower().endswith('.tif') or path.lower().endswith('.tiff'):
            with rasterio.open(path) as ds:
                m = ds.read(1)
        else:
            m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        m = (m > 0).astype(np.uint8)   # binary 0/1
        return m

    def _random_crop(self, img, mask, size):
        H,W,_ = img.shape
        if H < size or W < size:
            pad_h, pad_w = max(0, size-H), max(0, size-W)
            img  = cv2.copyMakeBorder(img,  0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            H,W,_ = img.shape
        y = random.randint(0, H-size)
        x = random.randint(0, W-size)
        return img[y:y+size, x:x+size], mask[y:y+size, x:x+size]

    '''def _augs(self, img, mask):
        # only for training
        if self.augment:
            if random.random() < 0.5:
                img  = np.ascontiguousarray(np.fliplr(img))
                mask = np.ascontiguousarray(np.fliplr(mask))
            if random.random() < 0.5:
                img  = np.ascontiguousarray(np.flipud(img))
                mask = np.ascontiguousarray(np.flipud(mask))
        return img, mask'''

    def _augs(self, img, mask):
        if self.augment:
            # Horizontal flip
            if random.random() < 0.5:
                img  = np.ascontiguousarray(np.fliplr(img))
                mask = np.ascontiguousarray(np.fliplr(mask))

            # Vertical flip
            if random.random() < 0.5:
                img  = np.ascontiguousarray(np.flipud(img))
                mask = np.ascontiguousarray(np.flipud(mask))

            # 90-degree rotations (0,90,180,270)
            if random.random() < 0.5:
                k = random.choice([1, 2, 3])  # 90,180,270
                img  = np.ascontiguousarray(np.rot90(img,  k, axes=(0,1)))
                mask = np.ascontiguousarray(np.rot90(mask, k, axes=(0,1)))

            # Small Gaussian noise (SAR-like speckle)
            if random.random() < 0.3:
                noise = np.random.normal(0, 0.05, img.shape).astype(np.float32)
                img = img + noise
                img = np.clip(img, -5.0, 5.0)  # keep in a reasonable range

            # Slight contrast/brightness jitter
            if random.random() < 0.3:
                alpha = 1.0 + 0.2 * (2*random.random() - 1)  # 0.8–1.2
                beta  = 0.1 * (2*random.random() - 1)        # -0.1–0.1
                img = alpha * img + beta
        return img, mask

    def __getitem__(self, idx):
        item = self.items[idx]
        img_path  = item['img']
        mask_path = item['mask']

        img  = self._read_tiff_2ch(img_path)
        mask = self._read_mask(mask_path)

        # normalize each channel (same as your training code)
        img = np.clip(img, -50, 5)
        img = (img - (-22.5)) / 12.5

        img, mask = self._augs(img, mask)
        if self.crop:
            img, mask = self._random_crop(img, mask, self.crop)

        img = np.transpose(img, (2,0,1))  # CHW
        return torch.from_numpy(img), torch.from_numpy(mask).long()


This is to display 2 channel tif images of the file of my choosing

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from rasterio.errors import RasterioIOError

tmp_ds = OilSpillDataset(all_items, crop_size=None, augment=False)

good_items = []
bad_items = []

'''for i, it in enumerate(all_items):
    try:
        x, y = tmp_ds[i]   # this will call _read_tiff_2ch + _read_mask
    except Exception as e:
        print(f"[BAD] idx={i}, img={it['img']} -> {type(e).__name__}: {e}")
        bad_items.append(it)
    else:
        good_items.append(it)

print(f"GOOD items: {len(good_items)}, BAD items: {len(bad_items)}")'''


# ============================================================
# Display a chosen 2-channel TIFF (by filename)
# ============================================================
def _to_numpy_2ch(x):
    """Convert torch/np input to numpy with shape (2,H,W)."""
    # torch tensor -> numpy
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()

    x = np.asarray(x)

    # Accept (2,H,W) or (H,W,2)
    if x.ndim == 3 and x.shape[0] == 2:
        return x
    if x.ndim == 3 and x.shape[-1] == 2:
        return np.transpose(x, (2, 0, 1))

    raise ValueError(f"Unexpected x shape for 2ch image: {x.shape}")


def _stretch01(img, lo=2, hi=98):
    """Percentile stretch to [0,1] for nicer viewing."""
    a, b = np.percentile(img, [lo, hi])
    img = (img - a) / (b - a + 1e-9)
    return np.clip(img, 0, 1)


def show_2ch_tif_by_name(ds, items, filename="00000.tif"):
    # find matching index by basename
    idx = None
    for i, it in enumerate(items):
        if os.path.basename(it["img"]) == filename:
            idx = i
            break

    if idx is None:
        raise FileNotFoundError(f"{filename} not found in all_items.")

    x, y = ds[idx]
    x2 = _to_numpy_2ch(x)  # (2,H,W)

    ch0 = _stretch01(x2[0])
    ch1 = _stretch01(x2[1])

    # Optional simple false-color composite for intuition
    # R = ch0, G = ch1, B = average
    rgb = np.stack([ch0, ch1, (ch0 + ch1) / 2], axis=-1)

    plt.figure(figsize=(14, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(ch0, cmap="gray")
    plt.title(f"{filename} - Channel 0")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(ch1, cmap="gray")
    plt.title(f"{filename} - Channel 1")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(rgb)
    plt.title(f"{filename} - False color (ch0,ch1)")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # quick stats
    print("Channel 0 stats:", float(ch0.min()), float(ch0.max()), float(ch0.mean()))
    print("Channel 1 stats:", float(ch1.min()), float(ch1.max()), float(ch1.mean()))

    return x, y, idx


# ---- Example usage ----
x_sel, y_sel, idx_sel = show_2ch_tif_by_name(tmp_ds, all_items, "00499.tif")
print("Selected index:", idx_sel)



=========THIS FOR 1 CHANNEL ONLY

In [ ]:
class OilSpillDataset(Dataset):
    def __init__(self, items, crop_size=512, augment=True, vv_only=False):
        """
        items: list of dicts with keys ['img', 'mask', 'subset', 'name']
        """
        self.items = items
        self.crop = crop_size
        self.augment = augment
        self.vv_only = vv_only   # <--- NEW FLAG

    def __len__(self):
        return len(self.items)

    def _read_tiff_2ch(self, path):
        with rasterio.open(path) as ds:
            arr = ds.read()  # (C,H,W) or (H,W)

        # Force to 2 bands [VV, VH] as before
        if arr.ndim == 3 and arr.shape[0] >= 2:
            arr = arr[:2]
        elif arr.ndim == 3 and arr.shape[0] == 1:
            arr = np.concatenate([arr, arr], 0)
        elif arr.ndim == 2:
            arr = np.stack([arr, arr], 0)
        else:
            raise ValueError(f"Unexpected TIFF shape {arr.shape} for {path}")

        arr = np.transpose(arr, (1, 2, 0)).astype(np.float32)  # (H,W,2)
        return arr

    def _read_mask(self, path):
        if path.lower().endswith('.tif') or path.lower().endswith('.tiff'):
            with rasterio.open(path) as ds:
                m = ds.read(1)
        else:
            m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        m = (m > 0).astype(np.uint8)
        return m

    def _random_crop(self, img, mask, size):
        H, W, _ = img.shape
        if H < size or W < size:
            pad_h, pad_w = max(0, size-H), max(0, size-W)
            img  = cv2.copyMakeBorder(img,  0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            H, W, _ = img.shape
        y = random.randint(0, H-size)
        x = random.randint(0, W-size)
        return img[y:y+size, x:x+size], mask[y:y+size, x:x+size]

    def _augs(self, img, mask):
        if self.augment:
            if random.random() < 0.5:
                img  = np.ascontiguousarray(np.fliplr(img))
                mask = np.ascontiguousarray(np.fliplr(mask))
            if random.random() < 0.5:
                img  = np.ascontiguousarray(np.flipud(img))
                mask = np.ascontiguousarray(np.flipud(mask))
        return img, mask

    def __getitem__(self, idx):
        item = self.items[idx]
        img_path  = item['img']
        mask_path = item['mask']

        img  = self._read_tiff_2ch(img_path)   # (H,W,2)
        mask = self._read_mask(mask_path)

        # 🔹 NEW: keep VV only if requested (take channel 0)
        if self.vv_only:
            img = img[..., :1]                 # (H,W,1)

        # normalize each channel (same logic)
        img = np.clip(img, -50, 5)
        img = (img - (-22.5)) / 12.5

        img, mask = self._augs(img, mask)
        if self.crop:
            img, mask = self._random_crop(img, mask, self.crop)

        img = np.transpose(img, (2, 0, 1))     # (C,H,W), C=1 if vv_only else 2

        return torch.from_numpy(img), torch.from_numpy(mask).long()




In [ ]:
all_items = good_items


In [ ]:
#@title 🧠 Model + Loss (DeepLabV3-ResNet50)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#model = deeplabv3_resnet50(weights=None, num_classes=cfg.num_classes)  # 2 classes
model = deeplabv3_resnet101(weights=None, num_classes=cfg.num_classes)  # 2 classes

# Patch the backbone stem to take 2 channels instead of 3
old_conv = model.backbone.conv1   # Conv2d(3,64,7,2,3)
new_conv = nn.Conv2d(2, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Kaiming init (good default)
nn.init.kaiming_normal_(new_conv.weight, mode='fan_out', nonlinearity='relu')

# OPTIONAL: if you ever load 3ch pretrained weights, you can average 3->2 here.
# For weights=None (fresh training), kaiming is fine.

model.backbone.conv1 = new_conv
model = model.to(device)

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0, ignore_index=255):
        super().__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index
    def forward(self, logits, targets):
        # logits: (B,C,H,W), targets: (B,H,W) long
        probs = torch.softmax(logits, dim=1)[:,1]  # foreground prob
        mask = (targets!=self.ignore_index).float()
        targets = (targets==1).float()
        probs = probs*mask; targets = targets*mask
        inter = (probs*targets).sum(dim=(1,2))
        denom = probs.sum(dim=(1,2))+targets.sum(dim=(1,2))
        dice = (2*inter + self.smooth)/(denom + self.smooth) # self.smooth prevents denom to be zero
        return 1 - dice.mean()

ce = nn.CrossEntropyLoss()
dice = DiceLoss()

def loss_fn(logits, targets):
    return ce(logits, targets) + 0.5*dice(logits, targets)

In [ ]:
#@title ⚡ Optimizer/Scheduler
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)

In [ ]:
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100, last_epoch=38)

In [ ]:
#@title 📏 Metrics
def metrics(logits, y):
    with torch.no_grad():
        pred = torch.argmax(logits, dim=1)
        tp = ((pred==1) & (y==1)).sum().item()
        tn = ((pred==0) & (y==0)).sum().item()
        fp = ((pred==1) & (y==0)).sum().item()
        fn = ((pred==0) & (y==1)).sum().item()
        iou = tp / max(1, (tp+fp+fn))
        prec = tp / max(1, (tp+fp))
        rec  = tp / max(1, (tp+fn))
        f1 = 2*prec*rec/max(1e-8, (prec+rec)) if (prec+rec)>0 else 0.0
    return {'IoU': iou, 'F1': f1, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn}

In [ ]:
import numpy as np

def validate_with_calibration(
    model,
    val_loader,
    device,
    n_bins=10,
    ignore_index=255
):
    model.eval()
    ce_loss = 0.0
    total_px = 0

    # confusion
    TP = FP = TN = FN = 0

    # calibration stats
    bin_total = np.zeros(n_bins, dtype=np.float64)
    bin_conf_sum = np.zeros(n_bins, dtype=np.float64)
    bin_correct = np.zeros(n_bins, dtype=np.float64)
    uncertainty_sum = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)['out']  # (B,2,H,W)
            loss = ce(logits, y)      # re-use your CE part
            ce_loss += loss.item() * x.size(0)

            # probs and predictions
            probs = torch.softmax(logits, dim=1)[:,1]  # foreground prob
            preds = (probs >= 0.5).long()              # binary pred (0/1)

            # flatten
            p_flat = probs.detach().cpu().numpy().ravel()
            y_flat = y.detach().cpu().numpy().ravel()
            pr_flat = preds.detach().cpu().numpy().ravel()

            if ignore_index is not None:
                valid = (y_flat != ignore_index)
                p_flat = p_flat[valid]
                y_flat = y_flat[valid]
                pr_flat = pr_flat[valid]

            if p_flat.size == 0:
                continue

            total_px += p_flat.size

            # confusion
            TP += np.sum((pr_flat == 1) & (y_flat == 1))
            FP += np.sum((pr_flat == 1) & (y_flat == 0))
            TN += np.sum((pr_flat == 0) & (y_flat == 0))
            FN += np.sum((pr_flat == 0) & (y_flat == 1))

            # calibration bins
            bin_idx = np.minimum((p_flat * n_bins).astype(int), n_bins-1)
            for b in range(n_bins):
                m = (bin_idx == b)
                if not np.any(m):
                    continue
                bin_total[b]    += m.sum()
                bin_conf_sum[b] += p_flat[m].sum()
                bin_correct[b]  += np.sum(pr_flat[m] == y_flat[m])

            # uncertainty p(1-p)
            uncertainty_sum += np.sum(p_flat * (1.0 - p_flat))

    # normalize losses & metrics
    ce_loss /= max(1, len(val_loader.dataset))

    IoU = TP / max(1, (TP + FP + FN))
    precision = TP / max(1, (TP + FP))
    recall = TP / max(1, (TP + FN))
    if precision + recall > 0:
        F1 = 2 * precision * recall / (precision + recall)
    else:
        F1 = 0.0

    FAR = FP / max(1, (FP + TN))

    # ECE + reliability diagram
    ece = 0.0
    bin_acc = np.zeros(n_bins, dtype=np.float64)
    bin_conf = np.zeros(n_bins, dtype=np.float64)
    if total_px > 0:
        for b in range(n_bins):
            if bin_total[b] == 0:
                bin_acc[b] = np.nan
                bin_conf[b] = np.nan
                continue
            bin_acc[b] = bin_correct[b] / bin_total[b]
            bin_conf[b] = bin_conf_sum[b] / bin_total[b]
            weight = bin_total[b] / total_px
            ece += weight * abs(bin_acc[b] - bin_conf[b])

    uncertainty_var = uncertainty_sum / max(1, total_px)

    metrics_out = {
        'loss': ce_loss,
        'IoU': float(IoU),
        'F1': float(F1),
        'FAR': float(FAR),
        'ECE': float(ece),
        'uncertainty_var': float(uncertainty_var),
        'TP': int(TP),
        'FP': int(FP),
        'TN': int(TN),
        'FN': int(FN),
        'bin_conf': bin_conf,
        'bin_acc': bin_acc,
        'bin_total': bin_total,
    }
    return metrics_out


In [ ]:
def plot_reliability_diagram(bin_conf, bin_acc, title='Reliability Diagram'):
    import matplotlib.pyplot as plt
    mask = ~np.isnan(bin_conf) & ~np.isnan(bin_acc)
    bc = bin_conf[mask]
    ba = bin_acc[mask]

    plt.figure(figsize=(4,4))
    plt.plot([0,1],[0,1],'--',label='Perfect')
    plt.plot(bc, ba, marker='o', label='Model')
    plt.xlabel('Predicted probability')
    plt.ylabel('Empirical accuracy')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
import rasterio
import numpy as np

one_img = "/content/drive/MyDrive/Sentinel SAR datasets/Oil/01289.tif"

with rasterio.open(one_img) as ds:
    print("bands:", ds.count, "size:", ds.width, ds.height, "dtype:", ds.dtypes)
    arr = ds.read()  # full read
    print("read shape:", arr.shape)


this is old - for VV+VH

In [ ]:
def build_model(cfg):
    #m = deeplabv3_resnet50(weights=None, num_classes=cfg.num_classes)
    m = deeplabv3_resnet101(weights=None, num_classes=cfg.num_classes)
    new_conv = nn.Conv2d(2, 64, kernel_size=7, stride=2, padding=3, bias=False)
    nn.init.kaiming_normal_(new_conv.weight, mode='fan_out', nonlinearity='relu')
    m.backbone.conv1 = new_conv
    return m


this is new - for VV

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50
import torch.nn as nn

def build_model(cfg):
    model = deeplabv3_resnet50(weights=None, num_classes=cfg.num_classes)

    # 1-channel input for VV-only
    model.backbone.conv1 = nn.Conv2d(
        in_channels=1,
        out_channels=64,
        kernel_size=7,
        stride=2,
        padding=3,
        bias=False
    )
    return model


In [ ]:
x, y = next(iter(train_loader))
print(x.shape, y.shape)
# Expect something like: torch.Size([4, 1, 512, 512]) torch.Size([4, 512, 512])


In [ ]:
# Example: test set = oil-only test tiles
test_items = test_oil_items       # or combine: test_oil_items + test_no_oil_items + test_look_items

test_ds = OilSpillDataset(
    test_items,
    crop_size=None,   # full tiles for test
    augment=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


🚀 Train

In [ ]:


from sklearn.model_selection import KFold
from torch.amp import GradScaler, autocast
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

kfold = KFold(n_splits=cfg.k_folds, shuffle=True, random_state=cfg.seed)
fold_summaries = []

#VV-only run
#cfg.in_channels = 1
#RESUME = False

#VV+VH- run
cfg.in_channels = 2
#RESUME = False

RESUME = True  # set False if you ever want to force retraining

for fold, (train_idx, val_idx) in enumerate(kfold.split(all_items), start=5):
    print('='*80)
    print(f'📚 Fold {fold}/{cfg.k_folds}')
    print('='*80)

    train_items = [all_items[i] for i in train_idx]
    val_items   = [all_items[i] for i in val_idx]

    train_ds = OilSpillDataset(train_items, crop_size=cfg.crop_size, augment=True)
    val_ds   = OilSpillDataset(val_items,   crop_size=cfg.crop_size, augment=False)

    #train_ds = OilSpillDataset(train_items, crop_size=512, augment=True,  vv_only=True)
    #val_ds   = OilSpillDataset(val_items,   crop_size=512, augment=False, vv_only=True)

    # Balanced sampler for training
    target_probs = {
        'oil': 0.5,
        'no_oil': 0.25,
        'lookalike': 0.25
    }
    #train_sampler = BalancedOilSampler(train_items, target_probs=target_probs)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, #sampler=train_sampler,   # <-- use sampler
                              shuffle=True, # <-- True when not using use sampler
                              num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size,
                              shuffle=False, num_workers=2, pin_memory=True)

    #ckpt_path = os.path.join(cfg.out_dir, f'deeplabv3_resnet50_oil_fold{fold}_best.pt')
    ckpt_path = os.path.join(cfg.out_dir, f'deeplabv3_resnet101_oil_fold{fold}_best.pt')
    #ckpt_path = os.path.join(cfg.out_dir, "deeplabv3_resnet101_oil_fold4_best.pt")
    #ckpt_path = ft_ckpt_path
    ft_ckpt_path = os.path.join(cfg.out_dir,    f'deeplabv3_resnet101_oil_fold{fold}_continued_best.pt')
    ckpt_path = ft_ckpt_path

    # --- model / optimizer / scheduler ---
    model = build_model(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    #sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)

    scaler = GradScaler('cuda', enabled=cfg.amp)

    # --- default start (fresh) ---
    start_epoch = 1
    best_iou = -1.0
    best_metrics = None

    # --- RESUME: if we already have a best checkpoint for this fold ---
    if RESUME and os.path.exists(ckpt_path):
        print(f"🔁 Resuming fold {fold} from {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)


        # load weights
        state = ckpt.get('state_dict', ckpt)
        state = {k.replace('module.', ''): v for k, v in state.items()}
        model.load_state_dict(state, strict=True)

        # get last best epoch & IoU
        prev_epoch = ckpt.get('epoch', 0)
        best_metrics = ckpt.get('metrics', {})
        best_iou = best_metrics.get('IoU', -1.0)
        #best_iou = 0.5
        start_epoch = prev_epoch + 1

        print(f"  last best epoch={prev_epoch}, best IoU={best_iou:.4f}")

        # Optional: if we already finished all epochs for this fold, skip it
        if start_epoch > cfg.epochs:
            print(f"  Fold {fold} already finished (start_epoch={start_epoch} > {cfg.epochs}), skipping.")
            fold_summaries.append({
                'fold': fold,
                'best_IoU':   best_iou,
                'best_F1':    best_metrics.get('F1', 0.0),
                'best_FAR':   best_metrics.get('FAR', 0.0),
                'best_ECE':   best_metrics.get('ECE', 0.0),
                'best_Uvar':  best_metrics.get('uncertainty_var', 0.0),
            })
            continue

    # --- main training loop for this fold ---
    for epoch in range(start_epoch, cfg.epochs+1):
        # -------- TRAIN --------
        model.train()
        tr_loss = 0.0
        for x,y in tqdm(train_loader, desc=f'Fold {fold} Epoch {epoch}/{cfg.epochs} [train]'):
            x,y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=cfg.amp):
                out = model(x)['out']
                loss = loss_fn(out, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tr_loss += loss.item() * x.size(0)
        tr_loss /= max(1, len(train_loader.dataset))
        sched.step()

        # -------- VALIDATE + CALIBRATION --------
        val_metrics = validate_with_calibration(
            model, val_loader, device, n_bins=cfg.n_bins, ignore_index=255
        )

        print(
            f"Fold {fold} Epoch {epoch:03d} | "
            f"train {tr_loss:.4f} | "
            f"val_loss {val_metrics['loss']:.4f} | "
            f"IoU {val_metrics['IoU']:.4f} | "
            f"F1 {val_metrics['F1']:.4f} | "
            f"FAR {val_metrics['FAR']:.4f} | "
            f"ECE {val_metrics['ECE']:.4f} | "
            f"U_var {val_metrics['uncertainty_var']:.4f}"
        )

        with open(cfg.log_file, mode="a", newline="") as f:
          writer = csv.writer(f)
          writer.writerow([
              datetime.now().isoformat(timespec="seconds"),
              fold,
              epoch,
              f"{tr_loss:.4f}",
              f"{val_metrics['loss']:.4f}",
              f"{val_metrics['IoU']:.4f}",
              f"{val_metrics['F1']:.4f}",
              f"{val_metrics['FAR']:.4f}",
              f"{val_metrics['ECE']:.4f}",
              f"{val_metrics['uncertainty_var']:.4f}"
          ])


        # track best by IoU
        if val_metrics['IoU'] > best_iou:
            best_iou = val_metrics['IoU']
            best_metrics = val_metrics
            torch.save({
                'state_dict': model.state_dict(),
                'cfg': vars(cfg),
                'fold': fold,
                'epoch': epoch,
                'metrics': best_metrics
            }, ft_ckpt_path)
            print(f"  ✅ New best fold {fold} IoU={best_iou:.4f} saved to {ft_ckpt_path}")

    # Store summary for this fold (best_metrics from latest / resumed run)
    fold_summaries.append({
        'fold': fold,
        'best_IoU':   best_metrics['IoU'],
        'best_F1':    best_metrics['F1'],
        'best_FAR':   best_metrics['FAR'],
        'best_ECE':   best_metrics['ECE'],
        'best_Uvar':  best_metrics['uncertainty_var'],
    })
    print(f"Fold {fold} best: IoU={best_metrics['IoU']:.4f}, "
          f"F1={best_metrics['F1']:.4f}, FAR={best_metrics['FAR']:.4f}, "
          f"ECE={best_metrics['ECE']:.4f}, Uvar={best_metrics['uncertainty_var']:.4f})")




This is new fine tune training which includes test set

In [ ]:
from sklearn.model_selection import KFold
from torch.amp import GradScaler, autocast
import os, csv
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# ------------------------------
# Fine-tune hyperparameters
# ------------------------------
FT_EPOCHS = 15          # how many fine-tune epochs per fold
FT_LR     = 1e-4        # smaller learning rate for fine-tuning

kfold = KFold(n_splits=cfg.k_folds, shuffle=True, random_state=cfg.seed)
fold_summaries = []

# VV+VH run
cfg.in_channels = 2

for fold, (train_idx, val_idx) in enumerate(kfold.split(all_items), start=5):
    print('='*80)
    print(f'📚 Fine-tune Fold {fold}/{cfg.k_folds}')
    print('='*80)

    # ---------------------------------
    # Build train / val datasets & loaders
    # ---------------------------------
    train_items = [all_items[i] for i in train_idx]
    val_items   = [all_items[i] for i in val_idx]

    train_ds = OilSpillDataset(train_items, crop_size=cfg.crop_size, augment=True)
    val_ds   = OilSpillDataset(val_items,   crop_size=cfg.crop_size, augment=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True
    )
    val_loader   = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # ---------------------------------
    # Checkpoints
    # ---------------------------------
    # Original best (from previous training run)
    ckpt_path        = os.path.join(cfg.out_dir, f'deeplabv3_resnet101_oil_fold{fold}_best.pt')
    # New fine-tuned best (by val IoU)
    ft_ckpt_path     = os.path.join(cfg.out_dir, f'deeplabv3_resnet101_oil_fold{fold}_continued_best.pt')
    # New best by test IoU
    ft_ckpt_path_test = os.path.join(cfg.out_dir, f'deeplabv3_resnet101_oil_fold{fold}_best_on_test.pt')
    ckpt_path        = ft_ckpt_path
    if not os.path.exists(ckpt_path):
        print(f"⚠️  No base checkpoint found for fold {fold}: {ckpt_path}. Skipping.")
        continue

    # ---------------------------------
    # Build model and load pre-trained weights
    # ---------------------------------
    model = build_model(cfg).to(device)

    print(f"🔁 Loading base checkpoint for fold {fold} from {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    state = ckpt.get('state_dict', ckpt)
    state = {k.replace('module.', ''): v for k, v in state.items()}
    model.load_state_dict(state, strict=True)

    prev_epoch   = ckpt.get('epoch', 0)
    prev_metrics = ckpt.get('metrics', {})

    # Start fine-tune from the previously best validation IoU
    best_val_iou     = prev_metrics.get('IoU', -1.0)
    best_val_metrics = prev_metrics
    best_test_iou    = -1.0   # we’ll track this only during FT

    print(f"  Base checkpoint: epoch={prev_epoch}, val IoU={best_val_iou:.4f}")

    # ---------------------------------
    # Optimizer & scheduler for FT
    # ---------------------------------
    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)

    scaler = GradScaler('cuda', enabled=cfg.amp)

    # ---------------------------------
    # Fine-tune loop (FT_EPOCHS only)
    # ---------------------------------
    for ft_epoch in range(1, FT_EPOCHS + 1):
        global_epoch = prev_epoch + ft_epoch  # for logging

        # -------- TRAIN --------
        model.train()
        tr_loss = 0.0
        for x, y in tqdm(train_loader, desc=f'Fold {fold} FT Epoch {ft_epoch}/{FT_EPOCHS} [train]'):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=cfg.amp):
                out = model(x)['out']
                loss = loss_fn(out, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tr_loss += loss.item() * x.size(0)

        tr_loss /= max(1, len(train_loader.dataset))
        sched.step()

        # -------- VALIDATION (fold val set) --------
        val_metrics = validate_with_calibration(
            model, val_loader, device, n_bins=cfg.n_bins, ignore_index=255
        )

        # -------- TEST SET EVAL (hold-out test_loader, NOT used for training) --------
        test_metrics = validate_with_calibration(
            model, test_loader, device, n_bins=cfg.n_bins, ignore_index=255
        )

        print(
            f"Fold {fold} FT-Epoch {ft_epoch:03d} (global {global_epoch}) | "
            f"train {tr_loss:.4f} | "
            f"val_loss {val_metrics['loss']:.4f} | "
            f"val_IoU {val_metrics['IoU']:.4f} | "
            f"val_F1 {val_metrics['F1']:.4f} | "
            f"FAR {val_metrics['FAR']:.4f} | "
            f"ECE {val_metrics['ECE']:.4f} | "
            f"U_var {val_metrics['uncertainty_var']:.4f} || "
            f"test_IoU {test_metrics['IoU']:.4f} | "
            f"test_F1 {test_metrics['F1']:.4f}"
        )

        # -------- LOG TO CSV --------
        with open(cfg.log_file, mode="a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.now().isoformat(timespec="seconds"),
                fold,
                global_epoch,              # log global epoch index
                f"{tr_loss:.4f}",
                f"{val_metrics['loss']:.4f}",
                f"{val_metrics['IoU']:.4f}",
                f"{val_metrics['F1']:.4f}",
                f"{val_metrics['FAR']:.4f}",
                f"{val_metrics['ECE']:.4f}",
                f"{val_metrics['uncertainty_var']:.4f}",
                f"{test_metrics['IoU']:.4f}",
                f"{test_metrics['F1']:.4f}",
            ])

        # -------- SAVE BEST BY VALIDATION IoU (including original best) --------
        if val_metrics['IoU'] > best_val_iou:
            best_val_iou     = val_metrics['IoU']
            best_val_metrics = val_metrics
            torch.save({
                'state_dict': model.state_dict(),
                'cfg': vars(cfg),
                'fold': fold,
                'epoch': global_epoch,
                'metrics': best_val_metrics
            }, ft_ckpt_path)
            print(f"  ✅ [VAL] New best fold {fold} IoU={best_val_iou:.4f} saved to {ft_ckpt_path}")

        # -------- SAVE BEST BY TEST IoU (separate file) --------
        if test_metrics['IoU'] > best_test_iou:
            best_test_iou = test_metrics['IoU']
            torch.save({
                'state_dict': model.state_dict(),
                'cfg': vars(cfg),
                'fold': fold,
                'epoch': global_epoch,
                'val_metrics': val_metrics,
                'test_metrics': test_metrics
            }, ft_ckpt_path_test)
            print(f"  ⭐ [TEST] New best fold {fold} test_IoU={best_test_iou:.4f} saved to {ft_ckpt_path_test}")

    # ---------------------------------
    # Store summary for this fold
    # ---------------------------------
    if best_val_metrics is None:
        best_val_metrics = {'IoU': 0.0, 'F1': 0.0, 'FAR': 0.0, 'ECE': 0.0, 'uncertainty_var': 0.0}

    fold_summaries.append({
        'fold': fold,
        'best_val_IoU':  best_val_metrics['IoU'],
        'best_val_F1':   best_val_metrics['F1'],
        'best_val_FAR':  best_val_metrics['FAR'],
        'best_val_ECE':  best_val_metrics['ECE'],
        'best_val_Uvar': best_val_metrics['uncertainty_var'],
        'best_test_IoU': best_test_iou,
    })

    print(
        f"Fold {fold} FINAL best VAL: IoU={best_val_metrics['IoU']:.4f}, "
        f"F1={best_val_metrics['F1']:.4f}, FAR={best_val_metrics['FAR']:.4f}, "
        f"ECE={best_val_metrics['ECE']:.4f}, Uvar={best_val_metrics['uncertainty_var']:.4f}"
    )
    print(f"Fold {fold} FINAL best TEST IoU: {best_test_iou:.4f}")


This is to clear memory

In [ ]:
import gc, torch

# Try to free any old models / tensors
gc.collect()
torch.cuda.empty_cache()


In [ ]:
for name in list(globals().keys()):
    if name.startswith('model') or name in ['opt', 'sched', 'scaler']:
        try:
            del globals()[name]
        except:
            pass

gc.collect()
torch.cuda.empty_cache()


THIS IS ADDED ON 06/01/2026 FROM GROK AS AN IMPROVEMENT IN BUILD MODEL

This is for fine tuning

In [ ]:
# ============================
# 🔧 Fine-tune best 2-ch DeepLabV3 model + log CSV
# ============================
import os, gc, csv, torch
from datetime import datetime
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.model_selection import KFold

gc.collect()
torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# ----------------------------
# 1) Choose which fold to fine-tune from
# ----------------------------
BEST_FOLD = 4  # <-- set this to the fold that gave you the best IoU

ckpt_path = os.path.join(
    cfg.out_dir,
    f"deeplabv3_resnet101_oil_fold{BEST_FOLD}_best.pt"
)
print("Loading base checkpoint:", ckpt_path)

# Build same 2-ch DeepLab model and load weights
model = build_model(cfg).to(device)   # your existing 2-channel builder

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
state = ckpt.get("state_dict", ckpt)


state = {k.replace("module.", ""): v for k, v in state.items()}
model.load_state_dict(state, strict=True)

# Replace this block:
# state = {k.replace("module.", ""): v for k, v in state.items()}
# model.load_state_dict(state, strict=True)

# With this:
#state = ckpt.get("state_dict", ckpt)
#state = {k.replace("module.", ""): v for k, v in state.items()}

# Load only the weights that match in name and shape
#missing_keys, unexpected_keys = model.load_state_dict(state, strict=False)

#print("Loaded old checkpoint with strict=False")
#print(f"   Missing keys (new layers): {len(missing_keys)}")
#print(f"   Unexpected keys (removed/old layers): {len(unexpected_keys)}")

# ----------------------------
# 2) Rebuild the BEST_FOLD validation split for validation metrics
# ----------------------------
kfold = KFold(n_splits=cfg.k_folds, shuffle=True, random_state=cfg.seed)

val_items = None
for fold, (train_idx, val_idx) in enumerate(kfold.split(all_items), start=1):
    if fold == BEST_FOLD:
        val_items = [all_items[i] for i in val_idx]
        break

assert val_items is not None, "Could not reconstruct validation items for BEST_FOLD."

val_ds = OilSpillDataset(
    val_items,
    crop_size=cfg.crop_size,  # you can also use None for full tiles
    augment=False
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ----------------------------
# 3) Fine-tune dataset (use ALL items, with maybe smaller crop)
# ----------------------------
cfg.crop_size = 384  # smaller crop if memory is tight (optional)

ft_ds = OilSpillDataset(
    all_items,
    crop_size=cfg.crop_size,
    augment=True
)

FT_BATCH_SIZE = 8     # drop to 2 or 1 if you still hit OOM
FT_EPOCHS     = 15     # fine-tune, not full retrain
FT_LR         = 1e-4  # smaller LR than main training

ft_loader = DataLoader(
    ft_ds,
    batch_size=FT_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

opt   = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
scaler = GradScaler('cuda', enabled=cfg.amp)

# ----------------------------
# 4) Prepare a separate CSV log for fine-tuning
# ----------------------------
ft_log_file = os.path.join(
    cfg.out_dir,
    f"finetune_fold{BEST_FOLD}_log.csv"
)
if not os.path.isfile(ft_log_file):
    print(f"Creating fine-tune log file: {ft_log_file}")
    with open(ft_log_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "timestamp", "fold", "epoch", "train_loss",
            "val_loss", "IoU", "F1", "FAR", "ECE", "uncertainty_var"
        ])

# ----------------------------
# 5) Fine-tune loop with validation + CSV logging
# ----------------------------
for epoch in range(1, FT_EPOCHS + 1):
    print(f"\n[Fine-tune] Epoch {epoch}/{FT_EPOCHS}")
    model.train()
    ft_loss = 0.0

    for x, y in tqdm(ft_loader, desc=f"FT Epoch {epoch}/{FT_EPOCHS} [train]"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=cfg.amp):
            out  = model(x)['out']
            #changed 06/01/2026
            #out = model(x)
            loss = loss_fn(out, y)  # same loss_fn you use for main training

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        ft_loss += loss.item() * x.size(0)

    ft_loss /= max(1, len(ft_loader.dataset))
    sched.step()
    print(f"[Fine-tune] train_loss={ft_loss:.4f}")

    # ---- VALIDATION on BEST_FOLD val set ----
    val_metrics = validate_with_calibration(
        model, val_loader, device,
        n_bins=cfg.n_bins,
        ignore_index=255
    )

    print(
        f"[Fine-tune] val_loss={val_metrics['loss']:.4f} | "
        f"IoU={val_metrics['IoU']:.4f} | "
        f"F1={val_metrics['F1']:.4f} | "
        f"FAR={val_metrics['FAR']:.4f} | "
        f"ECE={val_metrics['ECE']:.4f} | "
        f"U_var={val_metrics['uncertainty_var']:.4f}"
    )

    # ---- append to CSV ----
    with open(ft_log_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().isoformat(timespec="seconds"),
            BEST_FOLD,
            epoch,
            f"{ft_loss:.4f}",
            f"{val_metrics['loss']:.4f}",
            f"{val_metrics['IoU']:.4f}",
            f"{val_metrics['F1']:.4f}",
            f"{val_metrics['FAR']:.4f}",
            f"{val_metrics['ECE']:.4f}",
            f"{val_metrics['uncertainty_var']:.4f}",
        ])

# ----------------------------
# 6) Save final fine-tuned checkpoint
# ----------------------------
ft_ckpt_path = os.path.join(
    cfg.out_dir,
    f"deeplabv3_resnet101_oil_finetuned_from_fold{BEST_FOLD}.pt"
)
torch.save({
    "state_dict": model.state_dict(),
    "cfg": vars(cfg),
    "epoch": FT_EPOCHS,
    "source_fold": BEST_FOLD,
}, ft_ckpt_path)

print("✅ Fine-tuned model saved to:", ft_ckpt_path)
print("✅ Fine-tune metrics logged to:", ft_log_file)


Lets test foor inference

This is for single test set

In [ ]:
test_oil_items = [i for i in all_items if i['subset'] == 'test_oil']
test_no_oil_items = [i for i in all_items if i['subset'] == 'test_no_oil']
test_look_items = [i for i in all_items if i['subset'] == 'test_look']
print(test_oil_items[1])

In [ ]:
#for 3 ====  ckpt_path = "/content/drive/My Drive/oil_spill_checkpoints/periments/deepLabV3/deeplabv3_resnet101_oil_fold3_continued_best.pt"
ckpt_path = "/content/drive/My Drive/oil_spill_checkpoints/periments/deepLabV3/deeplabv3_resnet101_oil_fold5_continued_best.pt"   #for 5 ====
#ckpt_path = "/content/drive/My Drive/oil_spill_checkpoints/periments/deepLabV3/deeplabv3_resnet50_oil_fold3_best.pt"
# for 2 =====  ckpt_path = "/content/drive/My Drive/oil_spill_checkpoints/periments/deepLabV3/deeplabv3_resnet101_oil_only_fold2_best.pt"
#ckpt_path = "/content/drive/My Drive/oil_spill_checkpoints/periments/deepLabV3/deeplabv3_resnet101_oil_only_fold1_best.pt"
model = build_model(cfg).to(device)
ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
state = ckpt.get("state_dict", ckpt)
state = {k.replace("module.", ""): v for k,v in state.items()}
model.load_state_dict(state, strict=True)
model.eval()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, jaccard_score

def compute_iou_f1(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    iou = jaccard_score(y_true, y_pred, average="binary")
    f1 = f1_score(y_true, y_pred, average="binary")
    return iou, f1


# ----- Expected Calibration Error -----
def compute_ece(probs, labels, n_bins=15):
    """
    probs: (H,W) probability map 0-1
    labels: (H,W) GT 0/1
    """
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        avg_conf = probs[mask].mean()
        avg_acc  = labels[mask].mean()
        ece += (mask.sum() / probs.size) * abs(avg_conf - avg_acc)
    return ece


def plot_reliability_diagram(probs, labels, n_bins=15):
    bins = np.linspace(0, 1, n_bins+1)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    accs = []
    confs = []
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            accs.append(np.nan)
            confs.append(np.nan)
        else:
            confs.append(probs[mask].mean())
            accs.append(labels[mask].mean())

    plt.figure(figsize=(4,4))
    plt.plot([0,1],[0,1],'k--',label="Perfect calibration")
    plt.plot(confs, accs, marker='o')
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram")
    plt.grid(True)
    plt.show()


def compute_far(y_true, y_pred):
    """
    False Alarm Rate = FP / (FP + TN)
    y_true, y_pred: (H,W) binary 0/1
    """
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    fp = np.sum((y_pred == 1) & (y_true == 0))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    denom = fp + tn
    if denom == 0:
        return 0.0
    return fp / denom


def viz_batch_metrics(x, y, pred_logits, thr=0.3, show_metrics=True):
    x = x.detach().cpu().numpy()
    if y.ndim == 4: y = y.squeeze(1)
    y = y.detach().cpu().numpy()

    # 1-channel sigmoid vs 2-class softmax
    if pred_logits.shape[1] == 1:
        probs = torch.sigmoid(pred_logits).squeeze(1).cpu().numpy()  # B,H,W
        pred  = (probs > thr).astype(np.uint8)
    else:
        probs = torch.softmax(pred_logits, dim=1)[:,1,:,:].cpu().numpy()  # class-1 prob
        pred  = torch.argmax(pred_logits, dim=1).cpu().numpy()            # B,H,W

    b = min(3, x.shape[0])

    for i in range(b):
        vv = x[i,0]; vh = x[i,1]
        vv_norm = (vv - vv.min())/(vv.max()-vv.min()+1e-6)
        vh_norm = (vh - vh.min())/(vh.max()-vh.min()+1e-6)

        # ----- images -----
        plt.figure(figsize=(14,4))
        plt.subplot(1,5,1); plt.imshow(vv_norm, cmap='gray'); plt.title('VV'); plt.axis('off')
        plt.subplot(1,5,2); plt.imshow(vh_norm, cmap='gray'); plt.title('VH'); plt.axis('off')
        plt.subplot(1,5,3); plt.imshow(y[i],  cmap='gray');  plt.title('GT Mask'); plt.axis('off')
        plt.subplot(1,5,4); plt.imshow(pred[i], cmap='gray');plt.title('Pred Mask'); plt.axis('off')
        plt.subplot(1,5,5); plt.imshow((pred[i]>0)&(y[i]==0), cmap='Reds')
        plt.title('False Positives'); plt.axis('off')
        plt.show()

        if show_metrics:
            iou, f1 = compute_iou_f1(y[i], pred[i])
            far     = compute_far(y[i], pred[i])
            ece     = compute_ece(probs[i], y[i])

            print(f"Image {i}:")
            print(f"  IoU: {iou:.4f}")
            print(f"  F1 : {f1:.4f}")
            print(f"  FAR: {far:.4f}")
            print(f"  ECE: {ece:.4f}")

            # reliability plot
            plot_reliability_diagram(probs[i], y[i])


In [ ]:
# Choose which test item to inspect
val_items = [test_look_items[53]]

# No cropping, no augmentation for qualitative inspection
val_ds = OilSpillDataset(val_items, crop_size=None, augment=False)

from torch.utils.data import DataLoader

model.eval()  # important: eval mode
with torch.no_grad():  # no gradients during evaluation
    for x, y in DataLoader(val_ds, batch_size=1, shuffle=False):
        x = x.to(device); y = y.to(device)
        out = model(x)['out']          # logits
        viz_batch_metrics(x, y, out, thr=0.5, show_metrics=True)
        break  # only one image in this dataset


This is for the whole test set inference

In [ ]:
from torch.utils.data import DataLoader
import numpy as np

def evaluate_dataset_miou_f1(model, dataset, device, thr=0.3, batch_size=3):
    """
    Compute mean IoU and mean F1 over all images in `dataset`.

    Assumes:
      - model(x)['out'] gives logits of shape (B,2,H,W)
      - masks are binary 0/1 (no ignore_index) with shape (B,H,W) or (B,1,H,W)
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()

    all_iou = []
    all_f1  = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            if y.ndim == 4:
                y = y.squeeze(1)          # (B,H,W)

            logits = model(x)['out']      # (B,2,H,W)
            probs  = torch.softmax(logits, dim=1)[:, 1]   # class-1 prob → (B,H,W)
            preds  = (probs > thr).long()                 # (B,H,W)

            y_np    = y.cpu().numpy()
            preds_np = preds.cpu().numpy()

            for i in range(preds_np.shape[0]):
                iou, f1 = compute_iou_f1(y_np[i], preds_np[i])
                all_iou.append(iou)
                all_f1.append(f1)

    miou = float(np.mean(all_iou)) if all_iou else 0.0
    mf1  = float(np.mean(all_f1))  if all_f1 else 0.0

    print(f"mIoU over {len(all_iou)} images: {miou:.4f}")
    print(f"mF1  over {len(all_f1)} images: {mf1:.4f}")
    return miou, mf1


In [ ]:
# all test oil scenes you prepared earlier
val_items = test_oil_items          # NOT just one index now

test_ds = OilSpillDataset(
    val_items,
    crop_size=None,   # full tiles for evaluation
    augment=False
)

miou, mf1 = evaluate_dataset_miou_f1(
    model,
    test_ds,
    device,
    thr=0.3,          # or your preferred threshold
    batch_size=3      # lower if GPU memory is tight
)


NEWW EVALUATION TO INCLUDE FAR, ECE AND UNCERTAINTY VAR

In [ ]:
def evaluate_dataset_miou_f1(
    model,
    dataset,
    device,
    thr=0.3,
    batch_size=3,
    n_bins=10,
    ignore_index=None
):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()

    all_iou = []
    all_f1  = []

    TP = FP = TN = FN = 0
    total_px = 0

    bin_total    = np.zeros(n_bins, dtype=np.float64)
    bin_conf_sum = np.zeros(n_bins, dtype=np.float64)
    bin_correct  = np.zeros(n_bins, dtype=np.float64)
    uncertainty_sum = 0.0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            if y.ndim == 4:
                y = y.squeeze(1)

            logits = model(x)['out']                # (B,2,H,W)
            probs  = F.softmax(logits, dim=1)[:, 1] # P(oil)
            preds  = (probs > thr).long()

            y_np     = y.cpu().numpy()
            preds_np = preds.cpu().numpy()
            probs_np = probs.cpu().numpy()

            # ----- per-image IoU/F1 -----
            for i in range(preds_np.shape[0]):
                iou, f1 = compute_iou_f1(y_np[i], preds_np[i])
                all_iou.append(iou)
                all_f1.append(f1)

            # ----- global calibration stats -----
            p_flat  = probs_np.ravel()     # P(oil)
            y_flat  = y_np.ravel()
            pr_flat = preds_np.ravel()     # predicted class 0/1

            if ignore_index is not None:
                valid = (y_flat != ignore_index)
                p_flat  = p_flat[valid]
                y_flat  = y_flat[valid]
                pr_flat = pr_flat[valid]

            if p_flat.size == 0:
                continue

            # *** Correct confidence: prob of the predicted class ***
            conf_flat = np.where(pr_flat == 1, p_flat, 1.0 - p_flat)

            total_px += conf_flat.size

            TP += np.sum((pr_flat == 1) & (y_flat == 1))
            FP += np.sum((pr_flat == 1) & (y_flat == 0))
            TN += np.sum((pr_flat == 0) & (y_flat == 0))
            FN += np.sum((pr_flat == 0) & (y_flat == 1))

            # bins based on predicted confidence, not raw P(oil)
            bin_idx = np.minimum((conf_flat * n_bins).astype(int), n_bins - 1)
            for b in range(n_bins):
                m = (bin_idx == b)
                if not np.any(m):
                    continue
                bin_total[b]    += m.sum()
                bin_conf_sum[b] += conf_flat[m].sum()
                bin_correct[b]  += np.sum(pr_flat[m] == y_flat[m])

            # uncertainty using P(predicted class)*(1-P(predicted class))
            uncertainty_sum += np.sum(conf_flat * (1.0 - conf_flat))

    miou = float(np.mean(all_iou)) if all_iou else 0.0
    mf1  = float(np.mean(all_f1))  if all_f1 else 0.0

    FAR = FP / max(1, (FP + TN))

    ece = 0.0
    bin_acc  = np.zeros(n_bins, dtype=np.float64)
    bin_conf = np.zeros(n_bins, dtype=np.float64)
    if total_px > 0:
        for b in range(n_bins):
            if bin_total[b] == 0:
                bin_acc[b]  = np.nan
                bin_conf[b] = np.nan
                continue
            bin_acc[b]  = bin_correct[b] / bin_total[b]
            bin_conf[b] = bin_conf_sum[b] / bin_total[b]
            weight = bin_total[b] / total_px
            ece += weight * abs(bin_acc[b] - bin_conf[b])

    uncertainty_var = uncertainty_sum / max(1, total_px)

    print(f"mIoU over {len(all_iou)} images: {miou:.4f}")
    print(f"mF1  over {len(all_f1)} images: {mf1:.4f}")
    print(f"FAR  over all pixels       : {FAR:.4f}")
    print(f"ECE                        : {ece:.4f}")
    print(f"Uncertainty variance       : {uncertainty_var:.4f}")

    return {
        "mIoU": miou,
        "mF1": mf1,
        "FAR": float(FAR),
        "ECE": float(ece),
        "uncertainty_var": float(uncertainty_var),
        "TP": int(TP),
        "FP": int(FP),
        "TN": int(TN),
        "FN": int(FN),
        "bin_conf": bin_conf,
        "bin_acc": bin_acc,
        "bin_total": bin_total,
    }


RUN this for multi run uncertainty variance

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

def _enable_dropout_only(model):
    """
    Enable dropout layers during inference (MC Dropout) while keeping BatchNorm in eval.
    Call model.eval() first, then this function.
    """
    dropout_types = (
        torch.nn.Dropout,
        torch.nn.Dropout2d,
        torch.nn.Dropout3d,
        torch.nn.AlphaDropout,
        torch.nn.FeatureAlphaDropout,
    )
    for m in model.modules():
        if isinstance(m, dropout_types):
            m.train()

def evaluate_dataset_miou_f1_mc_dropout(
    model,
    dataset,
    device,
    thr=0.3,
    batch_size=3,
    n_bins=10,
    ignore_index=None,
    mc_passes=20,                 # number of stochastic forward passes
    mc_unbiased_var=False         # False => population var (divide by T), True => unbiased (T-1)
):
    """
    Computes:
      - mIoU, mF1, FAR, ECE (same behavior as your original, but using mean probs across MC passes)
      - mc_uncertainty_var: mean pixel-wise variance of P(oil) across MC passes (epistemic)
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    # Keep BatchNorm, etc. in eval mode
    model.eval()

    # Turn ON dropout only (MC Dropout)
    if mc_passes and mc_passes > 1:
        _enable_dropout_only(model)

    all_iou = []
    all_f1  = []

    TP = FP = TN = FN = 0
    total_px = 0

    bin_total    = np.zeros(n_bins, dtype=np.float64)
    bin_conf_sum = np.zeros(n_bins, dtype=np.float64)
    bin_correct  = np.zeros(n_bins, dtype=np.float64)

    # MC variance accumulator
    mc_var_sum = 0.0  # sum of per-pixel var(P(oil))
    mc_var_px  = 0    # count of valid pixels included

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            if y.ndim == 4:
                y = y.squeeze(1)  # (B,H,W)

            # --- MC Dropout: compute mean probs and var probs (P(oil)) across passes ---
            if mc_passes and mc_passes > 1:
                # Welford online mean/variance for probs (B,H,W)
                mean = None
                M2   = None
                for t in range(mc_passes):
                    logits_t = model(x)['out']                 # (B,2,H,W)
                    probs_t  = F.softmax(logits_t, dim=1)[:,1] # (B,H,W) P(oil)

                    if mean is None:
                        mean = probs_t.clone()
                        M2   = torch.zeros_like(probs_t)
                    else:
                        delta = probs_t - mean
                        mean = mean + delta / (t + 1)
                        delta2 = probs_t - mean
                        M2 = M2 + delta * delta2

                probs = mean  # mean P(oil) used for preds + metrics

                if mc_passes > 1:
                    denom = (mc_passes - 1) if mc_unbiased_var else mc_passes
                    var_map = M2 / max(1, denom)  # (B,H,W) variance of P(oil)
                else:
                    var_map = torch.zeros_like(probs)

            else:
                # single pass fallback (no MC)
                logits = model(x)['out']
                probs  = F.softmax(logits, dim=1)[:,1]
                var_map = torch.zeros_like(probs)

            preds = (probs > thr).long()

            # Move to CPU numpy for IoU/F1, bins (keep same structure as yours)
            y_np     = y.detach().cpu().numpy()
            preds_np = preds.detach().cpu().numpy()
            probs_np = probs.detach().cpu().numpy()
            var_np   = var_map.detach().cpu().numpy()

            # ----- per-image IoU/F1 -----
            for i in range(preds_np.shape[0]):
                iou, f1 = compute_iou_f1(y_np[i], preds_np[i])  # assumes your function exists
                all_iou.append(iou)
                all_f1.append(f1)

            # ----- global calibration + FAR stats -----
            p_flat  = probs_np.ravel()   # mean P(oil)
            y_flat  = y_np.ravel()
            pr_flat = preds_np.ravel()   # predicted class 0/1
            v_flat  = var_np.ravel()     # var(P(oil))

            if ignore_index is not None:
                valid = (y_flat != ignore_index)
                p_flat  = p_flat[valid]
                y_flat  = y_flat[valid]
                pr_flat = pr_flat[valid]
                v_flat  = v_flat[valid]

            if p_flat.size == 0:
                continue

            # Confidence of the predicted class (same as your logic)
            conf_flat = np.where(pr_flat == 1, p_flat, 1.0 - p_flat)

            total_px += conf_flat.size

            TP += np.sum((pr_flat == 1) & (y_flat == 1))
            FP += np.sum((pr_flat == 1) & (y_flat == 0))
            TN += np.sum((pr_flat == 0) & (y_flat == 0))
            FN += np.sum((pr_flat == 0) & (y_flat == 1))

            # Reliability bins (same as your logic)
            bin_idx = np.minimum((conf_flat * n_bins).astype(int), n_bins - 1)
            for b in range(n_bins):
                m = (bin_idx == b)
                if not np.any(m):
                    continue
                bin_total[b]    += m.sum()
                bin_conf_sum[b] += conf_flat[m].sum()
                bin_correct[b]  += np.sum(pr_flat[m] == y_flat[m])

            # ----- MC uncertainty: variance of P(oil) across passes -----
            # This is the "true multi-inference variance" (epistemic proxy).
            mc_var_sum += float(np.sum(v_flat))
            mc_var_px  += int(v_flat.size)

    miou = float(np.mean(all_iou)) if all_iou else 0.0
    mf1  = float(np.mean(all_f1))  if all_f1 else 0.0
    FAR  = FP / max(1, (FP + TN))

    # ECE from bins (same as your logic)
    ece = 0.0
    bin_acc  = np.zeros(n_bins, dtype=np.float64)
    bin_conf = np.zeros(n_bins, dtype=np.float64)
    if total_px > 0:
        for b in range(n_bins):
            if bin_total[b] == 0:
                bin_acc[b]  = np.nan
                bin_conf[b] = np.nan
                continue
            bin_acc[b]  = bin_correct[b] / bin_total[b]
            bin_conf[b] = bin_conf_sum[b] / bin_total[b]
            weight = bin_total[b] / total_px
            ece += weight * abs(bin_acc[b] - bin_conf[b])

    mc_uncertainty_var = mc_var_sum / max(1, mc_var_px)

    print(f"mIoU over {len(all_iou)} images: {miou:.4f}")
    print(f"mF1  over {len(all_f1)} images: {mf1:.4f}")
    print(f"FAR  over all pixels       : {FAR:.4f}")
    print(f"ECE                        : {ece:.4f}")
    print(f"MC Dropout var P(oil)      : {mc_uncertainty_var:.6f}  (passes={mc_passes})")

    return {
        "mIoU": miou,
        "mF1": mf1,
        "FAR": float(FAR),
        "ECE": float(ece),
        "mc_uncertainty_var": float(mc_uncertainty_var),
        "TP": int(TP),
        "FP": int(FP),
        "TN": int(TN),
        "FN": int(FN),
        "bin_conf": bin_conf,
        "bin_acc": bin_acc,
        "bin_total": bin_total,
    }


In [ ]:
# all test oil scenes you prepared earlier
val_items = test_oil_items          # NOT just one index now

test_ds = OilSpillDataset(
    val_items,
    crop_size=None,   # full tiles for evaluation
    augment=False
)

stats_test = evaluate_dataset_miou_f1_mc_dropout(
#stats_test = evaluate_dataset_miou_f1(
    model,
    test_ds,
    device,
    thr=0.1,
    batch_size=3,
    n_bins=10,
    ignore_index=255,
    mc_passes=20,                 # number of stochastic forward passes
    mc_unbiased_var=False         # False => population var (divide by T), True => unbiased (T-1)
)

print(stats_test["mIoU"], stats_test["FAR"], stats_test["ECE"])


=THIS IS TO TEST THE LOOK ALIKES DATASET (FALSE POSITIVES)

In [ ]:
import numpy as np
from torch.utils.data import DataLoader


def evaluate_lookalike_dataset(
    model,
    dataset,
    device,
    thr=0.3,
    batch_size=3
):
    """
    Evaluate a model on a dataset of look-alike (no-oil) scenes.

    Returns:
      stats: dict with
        - overall_far        : FP / (FP+TN) over all pixels
        - mean_far_per_img   : mean FAR across images
        - median_far_per_img : median FAR across images
        - max_far_per_img    : worst-case FAR across images
        - frac_imgs_with_fp  : fraction of images with any predicted oil
        - n_images           : number of images evaluated
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model.eval()

    total_fp = 0
    total_tn = 0
    n_images = 0
    imgs_with_any_fp = 0

    per_img_far = []
    per_img_pos_frac = []  # same as FAR when GT=0, but useful if some masks have 1s

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            # y: (B, H, W) (your dataset already returns 0/1 masks)
            if y.ndim == 4:
                y = y.squeeze(1)

            logits = model(x)['out']                  # (B,2,H,W)
            probs  = torch.softmax(logits, dim=1)[:,1]  # class-1 prob (B,H,W)
            preds  = (probs > thr).long()             # (B,H,W)

            y_np   = y.cpu().numpy()
            pr_np  = preds.cpu().numpy()

            B = pr_np.shape[0]
            for i in range(B):
                gt  = y_np[i].ravel()
                pr  = pr_np[i].ravel()

                fp = np.sum((pr == 1) & (gt == 0))
                tn = np.sum((pr == 0) & (gt == 0))

                n_images += 1
                if fp > 0:
                    imgs_with_any_fp += 1

                denom = fp + tn
                if denom == 0:
                    far = 0.0
                    pos_frac = 0.0
                else:
                    far = fp / denom
                    pos_frac = fp / denom

                per_img_far.append(far)
                per_img_pos_frac.append(pos_frac)

                total_fp += fp
                total_tn += tn

    overall_denom = total_fp + total_tn
    overall_far = total_fp / overall_denom if overall_denom > 0 else 0.0

    mean_far = float(np.mean(per_img_far)) if per_img_far else 0.0
    median_far = float(np.median(per_img_far)) if per_img_far else 0.0
    max_far = float(np.max(per_img_far)) if per_img_far else 0.0

    frac_imgs_with_fp = imgs_with_any_fp / n_images if n_images > 0 else 0.0

    stats = {
        "overall_far": overall_far,
        "mean_far_per_img": mean_far,
        "median_far_per_img": median_far,
        "max_far_per_img": max_far,
        "frac_imgs_with_fp": frac_imgs_with_fp,
        "n_images": n_images,
    }

    print("===== Look-alike set (no-oil) evaluation =====")
    print(f"Images evaluated        : {n_images}")
    print(f"Overall FAR (all pixels): {overall_far:.6f}")
    print(f"Mean FAR per image      : {mean_far:.6f}")
    print(f"Median FAR per image    : {median_far:.6f}")
    print(f"Max FAR per image       : {max_far:.6f}")
    print(f"Images with any FP      : {imgs_with_any_fp} "
          f"({100*frac_imgs_with_fp:.2f}% of images)")
    print("==============================================")

    return stats


In [ ]:
val_items = test_look_items   # all look-alike tiles

look_ds = OilSpillDataset(
    val_items,
    crop_size=None,   # full tiles
    augment=False
)

stats_look = evaluate_lookalike_dataset(
    model,
    look_ds,
    device,
    thr=0.3,          # same threshold you use for IoU/F1
    batch_size=3
)


===========LOOK ALIKE TEST END

==============THIS IS TO TEST FALSE NEGATIVES ON DATASETS WITH OIL

In [ ]:
import numpy as np
from torch.utils.data import DataLoader

def evaluate_oil_dataset_fn(
    model,
    dataset,
    device,
    thr=0.3,
    batch_size=3
):
    """
    Evaluate performance and false negatives on an 'oil' dataset
    where GT masks contain oil (1) and background (0).

    Metrics:
      - mIoU, mF1 over full masks
      - FNR over *oil pixels only* (FN / (FN+TP) where gt==1)
      - per-image FNR_pos stats
      - fraction of images with any FN on oil pixels
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()

    all_iou = []
    all_f1  = []

    per_img_fnr_pos = []

    total_fn_pos = 0
    total_tp_pos = 0
    total_pos_px = 0

    imgs_with_any_fn_pos = 0
    n_images = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            if y.ndim == 4:
                y = y.squeeze(1)  # (B,H,W)

            logits = model(x)['out']                      # (B,2,H,W)
            probs  = torch.softmax(logits, dim=1)[:, 1]   # class-1 prob (B,H,W)
            preds  = (probs > thr).long()                 # (B,H,W)

            y_np  = y.cpu().numpy()
            pr_np = preds.cpu().numpy()

            B = pr_np.shape[0]
            for i in range(B):
                gt = y_np[i].ravel()
                pr = pr_np[i].ravel()

                # ---- IoU/F1 over the full mask ----
                iou, f1 = compute_iou_f1(gt, pr)
                all_iou.append(iou)
                all_f1.append(f1)

                # ---- False negatives on *oil pixels only* (gt==1) ----
                pos_mask = (gt == 1)
                pos_total = int(pos_mask.sum())

                # Some tiles might weirdly have no oil pixels -> skip FNR
                if pos_total == 0:
                    per_img_fnr_pos.append(np.nan)
                    continue

                tp_pos = int(((pr == 1) & pos_mask).sum())
                fn_pos = int(((pr == 0) & pos_mask).sum())

                n_images += 1
                if fn_pos > 0:
                    imgs_with_any_fn_pos += 1

                total_tp_pos += tp_pos
                total_fn_pos += fn_pos
                total_pos_px += pos_total

                fnr_pos = fn_pos / pos_total  # FN / (TP+FN) on oil pixels
                per_img_fnr_pos.append(fnr_pos)

    # ---- Aggregate ----
    miou = float(np.mean(all_iou)) if all_iou else 0.0
    mf1  = float(np.mean(all_f1))  if all_f1 else 0.0

    denom_pos = total_fn_pos + total_tp_pos
    overall_fnr_pos = total_fn_pos / denom_pos if denom_pos > 0 else 0.0

    per_img_fnr_pos = np.array(per_img_fnr_pos, dtype=np.float64)
    mean_fnr_pos   = float(np.nanmean(per_img_fnr_pos)) if per_img_fnr_pos.size > 0 else 0.0
    median_fnr_pos = float(np.nanmedian(per_img_fnr_pos)) if per_img_fnr_pos.size > 0 else 0.0
    max_fnr_pos    = float(np.nanmax(per_img_fnr_pos)) if per_img_fnr_pos.size > 0 else 0.0

    frac_imgs_with_fn_pos = imgs_with_any_fn_pos / n_images if n_images > 0 else 0.0

    print("===== Oil dataset: performance + false negatives on oil pixels =====")
    print(f"Images (with pos counted)   : {n_images}")
    print(f"mIoU (full masks)           : {miou:.4f}")
    print(f"mF1  (full masks)           : {mf1:.4f}")
    print("---- False negatives on oil pixels (gt==1) ----")
    print(f"Overall FNR_pos (all oil px): {overall_fnr_pos:.6f}")
    print(f"Mean FNR_pos per image      : {mean_fnr_pos:.6f}")
    print(f"Median FNR_pos per image    : {median_fnr_pos:.6f}")
    print(f"Max FNR_pos per image       : {max_fnr_pos:.6f}")
    print(f"Images with any FN on oil   : {imgs_with_any_fn_pos} "
          f"({100*frac_imgs_with_fn_pos:.2f}% of images)")
    print("====================================================================")

    return {
        "mIoU": miou,
        "mF1": mf1,
        "overall_fnr_pos": overall_fnr_pos,
        "mean_fnr_pos": mean_fnr_pos,
        "median_fnr_pos": median_fnr_pos,
        "max_fnr_pos": max_fnr_pos,
        "frac_imgs_with_fn_pos": frac_imgs_with_fn_pos,
        "n_images_pos_counted": n_images,
    }


In [ ]:
val_items = test_oil_items   # all oil-containing tiles

oil_ds = OilSpillDataset(
    val_items,
    crop_size=None,   # full tiles
    augment=False
)

stats_oil_fn = evaluate_oil_dataset_fn(
    model,
    oil_ds,
    device,
    thr=0.3,          # same threshold as for IoU/F1
    batch_size=3
)


==================END OF FALSE NEGATIVE TESTS

In [ ]:
import rasterio
import numpy as np
import torch

def load_tiff_2ch_for_inference(path, device, fill_value=-50.0):
    """
    Load 2-channel VV/VH TIFF for inference and handle NaNs/masked values.

    - Uses rasterio.read(masked=True) so nodata becomes masked.
    - Fills masked and NaN with a low backscatter (e.g. -50 dB).
    - Applies the same dB normalization as in training.
    """
    with rasterio.open(path) as ds:
        arr = ds.read(masked=True)  # (C,H,W) MaskedArray

    # Convert masked to normal ndarray and fill masked values
    if isinstance(arr, np.ma.MaskedArray):
        arr = arr.filled(fill_value=fill_value)

    # Also handle NaNs explicitly
    arr = np.nan_to_num(arr, nan=fill_value)

    # Enforce 2 bands like before
    if arr.ndim == 3 and arr.shape[0] >= 2:
        arr = arr[:2]
    elif arr.ndim == 3 and arr.shape[0] == 1:
        arr = np.concatenate([arr, arr], 0)
    elif arr.ndim == 2:
        arr = np.stack([arr, arr], 0)
    else:
        raise ValueError(f"Unexpected TIFF shape {arr.shape} for {path}")

    # (C,H,W) -> (H,W,C)
    img = np.transpose(arr, (1, 2, 0)).astype(np.float32)

    # ✅ same normalization as training data
    img = np.clip(img, -50, 5)
    img = (img - (-22.5)) / 12.5

    # (H,W,C) -> (C,H,W)
    img = np.transpose(img, (2, 0, 1))  # (2,H,W)

    x = torch.from_numpy(img).unsqueeze(0).to(device)  # (1,2,H,W)
    return x


In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

def viz_unlabeled_inference(x, pred_logits, thr=0.3):
    """
    x: (1,2,H,W) tensor (normalized)
    pred_logits: (1,C,H,W) tensor, C=2
    """
    x_np = x.detach().cpu().numpy()[0]        # (2,H,W)
    vv = x_np[0]
    vh = x_np[1]

    # normalize for display
    vv_norm = (vv - vv.min()) / (vv.max() - vv.min() + 1e-6)
    vh_norm = (vh - vh.min()) / (vh.max() - vh.min() + 1e-6)

    # probs + pred mask
    with torch.no_grad():
        probs = torch.softmax(pred_logits, dim=1)[0, 1]  # (H,W)
    probs_np = probs.cpu().numpy()
    pred_mask = (probs_np > thr).astype(np.uint8)

    plt.figure(figsize=(14,4))

    plt.subplot(1,4,1)
    plt.imshow(vv_norm, cmap='gray')
    plt.title("VV")
    plt.axis("off")

    plt.subplot(1,4,2)
    plt.imshow(vh_norm, cmap='gray')
    plt.title("VH")
    plt.axis("off")

    plt.subplot(1,4,3)
    plt.imshow(probs_np, cmap='viridis')
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title("Oil probability")
    plt.axis("off")

    plt.subplot(1,4,4)
    plt.imshow(pred_mask, cmap='gray')
    plt.title(f"Pred mask (thr={thr})")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# Path to your historical SAR TIFF (VV/VH)
historical_path = "/content/drive/MyDrive/Sentinel SAR datasets/historical_spills/S1A_IW_GRDH_1SDV_20240815T214714_20240815T214739_055229_06BB7A_4C54_oilprep.tif"

# 1) Load and preprocess
x = load_tiff_2ch_for_inference(historical_path, device)  # (1,2,H,W)

# 2) Run model in eval mode
model.eval()
with torch.no_grad():
    out = model(x)['out']  # (1,2,H,W) logits for background/oil

# 3) Visualize (VV, VH, prob, pred)
viz_unlabeled_inference(x, out, thr=0.3)


===========WORKING AND TESTED UP TO HERE ===============06/01/2026

In [ ]:
from torch.utils.data import DataLoader
import numpy as np

def evaluate_dataset_miou(model, dataset, device, thr=0.3, batch_size=3):
    """
    Loop over all items in `dataset` and compute mean IoU / F1.
    Assumes:
      - logits: (B, C, H, W) with C=2
      - targets: (B, H, W) with 0/1 labels
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()

    all_iou = []
    all_f1  = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)['out']                  # (B,2,H,W)
            probs  = torch.softmax(logits, dim=1)[:,1]  # foreground prob (B,H,W)
            preds  = (probs > thr).long()             # (B,H,W)

            viz_batch_metrics(x, y, logits, thr=0.3, show_metrics=True)

            # move to CPU numpy
            y_np    = y.cpu().numpy()
            preds_np = preds.cpu().numpy()

            # if masking with 255 is needed, handle here
            # (you can extend this block with IGNORE_INDEX if you want)

            # compute per-image IoU/F1
            if y_np.ndim == 4:       # just in case, squeeze channel dim
                y_np = y_np.squeeze(1)
            for i in range(preds_np.shape[0]):
                iou, f1 = compute_iou_f1(y_np[i], preds_np[i])
                all_iou.append(iou)
                all_f1.append(f1)

    miou = float(np.mean(all_iou)) if all_iou else 0.0
    mf1  = float(np.mean(all_f1))  if all_f1 else 0.0

    print(f"mIoU over {len(all_iou)} images: {miou:.4f}")
    print(f"mF1  over {len(all_f1)} images: {mf1:.4f}")
    return miou, mf1


In [ ]:
# e.g. evaluate a whole subset
val_items = test_oil_items          # or some slice/list you want
#val_items = test_no_oil_items
#val_items = test_look_items
val_ds = OilSpillDataset(
    val_items,
    crop_size=None,
    augment=False,
    #in_channels=cfg.in_channels
)

miou, mf1 = evaluate_dataset_miou(model, val_ds, device, thr=0.3, batch_size=3)


In [ ]:
import pandas as pd
df_folds = pd.DataFrame(fold_summaries)
print(df_folds)
print("Mean over folds:")
print(df_folds.mean(numeric_only=True))


In [ ]:
#@title 🚀 Train
scaler = GradScaler('cuda', enabled=cfg.amp)
best_iou, best_path = -1.0, os.path.join(cfg.out_dir, 'deeplabv3_resnet50_oil_best_2.pt')

for epoch in range(1, cfg.epochs+1):
    model.train()
    tr_loss = 0.0
    for x,y in tqdm(train_loader, desc=f'Epoch {epoch}/{cfg.epochs} [train]'):
        x,y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=cfg.amp):

            out = model(x)['out']
            loss = loss_fn(out, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tr_loss += loss.item()*x.size(0)
    tr_loss /= len(train_loader.dataset)
    sched.step()

    # val
    model.eval()
    va_loss = 0.0; all_m = []
    with torch.no_grad():
        for x,y in DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True):
            x,y = x.to(device), y.to(device)
            out = model(x)['out']
            loss = loss_fn(out, y)
            va_loss += loss.item()*x.size(0)
            all_m.append(metrics(out, y))
    va_loss /= max(1, len(val_ds))
    # aggregate metrics
    if all_m:
        iou = float(np.mean([m['IoU'] for m in all_m]))
        f1  = float(np.mean([m['F1']  for m in all_m]))
    else:
        iou=f1=0.0

    print(f'Epoch {epoch:03d} | train {tr_loss:.4f} | val {va_loss:.4f} | IoU {iou:.4f} | F1 {f1:.4f} | lr {sched.get_last_lr()[0]:.2e}')

    if iou > best_iou:
        best_iou = iou
        torch.save({'state_dict': model.state_dict(),
                    'cfg': vars(cfg),
                    'epoch': epoch,
                    'metrics': {'IoU': iou, 'F1': f1}}, best_path)
        print('  ✅ Saved:', best_path)

load best check point and train few more epochs

In [ ]:
# --- Resume from deeplabv3_resnet50_oil_best_2.pt and continue training ---
import os, torch, numpy as np
from torch.cuda.amp import GradScaler
from torch.utils.data import DataLoader

# paths
in_ckpt  = os.path.join(cfg.out_dir, 'deeplabv3_resnet101_oil_fold2_best.pt')  # load from here
out_ckpt = os.path.join(cfg.out_dir, 'deeplabv3_resnet101_oil_fold2_best_fine_tune1.pt')  # save new best here

# load weights (you saved as {'state_dict', 'cfg', 'epoch', 'metrics': {...}})
ckpt = torch.load(in_ckpt, map_location=device)
state = ckpt.get('state_dict', ckpt)
state = {k.replace('module.', ''): v for k, v in state.items()}  # safe if DP was used
model.load_state_dict(state, strict=True)

start_epoch = ckpt.get('epoch', 0) + 1
best_iou    = ckpt.get('metrics', {}).get('IoU', -1.0)
print(f"Resumed from {in_ckpt} @ epoch {start_epoch-1} (best IoU={best_iou:.4f})")

# fresh fine-tune LR (since you didn’t save optimizer state)
for g in opt.param_groups:
    g['lr'] = getattr(cfg, 'resume_lr', 3e-4)  # tweak as needed

# re-init scaler if needed
#scaler = GradScaler('cuda', enabled=cfg.amp)

AMP_ENABLED = bool(getattr(cfg, 'amp', True) and torch.cuda.is_available())
scaler = GradScaler(enabled=AMP_ENABLED)

EPOCHS_MORE = getattr(cfg, 'epochs_more', 50)   # how many extra epochs



for epoch in range(start_epoch, start_epoch + EPOCHS_MORE):
    # ---- train ----
    model.train()
    tr_loss = 0.0
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch}/{start_epoch+EPOCHS_MORE-1} [train]'):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        # prefer torch.autocast to avoid signature conflicts
        with torch.autocast(device_type='cuda', enabled=cfg.amp):
            out  = model(x)['out']
            loss = loss_fn(out, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tr_loss += loss.item() * x.size(0)
    tr_loss /= max(1, len(train_loader.dataset))
    # If you use ReduceLROnPlateau, move sched.step(iou) after validation
    sched.step()

    # ---- validate ----
    model.eval()
    va_loss = 0.0; all_m = []
    with torch.no_grad():
        for x, y in DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True):
            x, y = x.to(device), y.to(device)
            out  = model(x)['out']
            loss = loss_fn(out, y)
            va_loss += loss.item() * x.size(0)
            all_m.append(metrics(out, y))
    va_loss /= max(1, len(val_ds))
    iou = float(np.mean([m['IoU'] for m in all_m])) if all_m else 0.0
    f1  = float(np.mean([m['F1']  for m in all_m])) if all_m else 0.0

    print(f'Epoch {epoch:03d} | train {tr_loss:.4f} | val {va_loss:.4f} | IoU {iou:.4f} | F1 {f1:.4f} | lr {opt.param_groups[0]["lr"]:.2e}')

    # save new best to *_best_3.pt
    if iou > best_iou:
        best_iou = iou
        torch.save({'state_dict': model.state_dict(),
                    'cfg': vars(cfg),
                    'epoch': epoch,
                    'metrics': {'IoU': iou, 'F1': f1}}, out_ckpt)
        print('  ✅ Saved:', out_ckpt)


In [ ]:
import torch
import torch.nn as nn
#from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision.models.segmentation import deeplabv3_resnet101
def load_deeplab_model(ckpt_path, device, in_ch=2, num_classes=2):
    # 1) build model
    model = deeplabv3_resnet101(weights=None, num_classes=num_classes)

    # 2) patch first conv to accept 2 channels
    old_conv = model.backbone.conv1
    if old_conv.in_channels != in_ch:
        new_conv = nn.Conv2d(
            in_ch,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=(old_conv.bias is not None)
        )
        with torch.no_grad():
            avg_w = old_conv.weight.mean(dim=1, keepdim=True)
            new_conv.weight.copy_(avg_w.repeat(1, in_ch, 1, 1))
        model.backbone.conv1 = new_conv

    # 3) load checkpoint (IMPORTANT FIX)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    # handle different save formats
    if isinstance(ckpt, dict):
        if "model" in ckpt:
            sd = ckpt["model"]
        elif "state_dict" in ckpt:
            sd = ckpt["state_dict"]
        else:
            sd = ckpt
    else:
        sd = ckpt

    # remove "module." if saved from DataParallel
    sd = {k.replace("module.", ""): v for k, v in sd.items()}

    # 4) load weights
    model.load_state_dict(sd, strict=False)
    model.to(device)
    model.eval()
    return model


In [ ]:
#@title 👀 Visualize a few predictions
def viz_batch(x, y, pred):
    # x: B,2,H,W (normalized)
    x = x.cpu().numpy()
    y = y.cpu().numpy()
    pred = torch.argmax(pred, dim=1).cpu().numpy()
    b = min(3, x.shape[0])
    for i in range(b):
        vv = x[i,0]; vh = x[i,1]
        vv = (vv - vv.min())/(vv.max()-vv.min()+1e-6)
        vh = (vh - vh.min())/(vh.max()-vh.min()+1e-6)
        rgb = np.stack([vv, vh, vv*0.5+vh*0.5], axis=-1)
        plt.figure(figsize=(12,3))
        plt.subplot(1,4,1); plt.imshow(rgb); plt.title('VV/VH composite'); plt.axis('off')
        plt.subplot(1,4,2); plt.imshow(y[i], cmap='gray'); plt.title('GT mask'); plt.axis('off')
        plt.subplot(1,4,3); plt.imshow(pred[i], cmap='gray'); plt.title('Pred mask'); plt.axis('off')
        plt.subplot(1,4,4); plt.imshow((pred[i]>0)&(y[i]==0), cmap='Reds'); plt.title('FP (red)'); plt.axis('off')
        plt.show()

def viz_batch2(x, y, pred_logits, thr=0.3):
    # x: B,2,H,W  y: B,H,W or B,1,H,W  pred_logits: B,C,H,W (C=1 or 2)
    x = x.detach().cpu().numpy()
    if y.ndim == 4: y = y.squeeze(1)
    y = y.detach().cpu().numpy()

    # Handle 1-channel vs 2-class heads
    if pred_logits.shape[1] == 1:
        prob = torch.sigmoid(pred_logits)
        pred = (prob > thr).to(torch.uint8).squeeze(1).cpu().numpy()  # B,H,W
    else:
        pred = torch.argmax(pred_logits, dim=1).to(torch.uint8).cpu().numpy()  # B,H,W

    b = min(3, x.shape[0])
    for i in range(b):
        vv = x[i,0]; vh = x[i,1]
        vv = (vv - vv.min())/(vv.max()-vv.min()+1e-6)
        vh = (vh - vh.min())/(vh.max()-vh.min()+1e-6)
        rgb = np.stack([vv, vh, 0.5*vv+0.5*vh], axis=-1)

        plt.figure(figsize=(12,3))
        plt.subplot(1,4,1); plt.imshow(vv, cmap='gray'); plt.title('VV/VH composite'); plt.axis('off')
        plt.subplot(1,4,2); plt.imshow(y[i], cmap='gray'); plt.title('GT mask'); plt.axis('off')
        plt.subplot(1,4,3); plt.imshow(pred[i], cmap='gray'); plt.title('Pred mask'); plt.axis('off')
        plt.subplot(1,4,4); plt.imshow((pred[i]>0)&(y[i]==0), cmap='Reds'); plt.title('FP (red)'); plt.axis('off')
        plt.show()

ckpt_path = "/content/drive/MyDrive/oil_spill_checkpoints/periments/deepLabV3/deeplabv3_resnet101_oil_fold2_best.pt"

model = load_deeplab_model(
    ckpt_path,
    device=device,
    in_ch=2,                 # 2 channels (VV,VH)
    num_classes=cfg.num_classes
)


val_items   = [all_items[103]]


#val_ds   = OilSpillDataset(val_items,   crop_size=cfg.crop_size, augment=False)
val_ds   = OilSpillDataset(val_items,   crop_size=None, augment=False)
#with torch.no_grad():
#    for x,y in DataLoader(val_ds, batch_size=3, shuffle=True):
#        x = x.to(device); y = y.to(device)
#        out = model(x)['out']
#        viz_batch2(x, y, out)
#        break

This is an improvement from the cell above

In [ ]:
#---------------------------------------------
# 📌 METRICS
#---------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, jaccard_score

def compute_iou_f1(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    iou = jaccard_score(y_true, y_pred, average="binary")
    f1 = f1_score(y_true, y_pred, average="binary")
    return iou, f1


# ----- Expected Calibration Error -----
def compute_ece(probs, labels, n_bins=15):
    """
    probs: (H,W) probability map 0-1
    labels: (H,W) GT 0/1
    """
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        avg_conf = probs[mask].mean()
        avg_acc  = labels[mask].mean()
        ece += (mask.sum() / probs.size) * abs(avg_conf - avg_acc)
    return ece


def plot_reliability_diagram(probs, labels, n_bins=15):
    bins = np.linspace(0, 1, n_bins+1)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    accs = []
    confs = []
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            accs.append(np.nan)
            confs.append(np.nan)
        else:
            confs.append(probs[mask].mean())
            accs.append(labels[mask].mean())

    plt.figure(figsize=(4,4))
    plt.plot([0,1],[0,1],'k--',label="Perfect calibration")
    plt.plot(confs, accs, marker='o')
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram")
    plt.grid(True)
    plt.show()


#---------------------------------------------
# 📌 VISUALIZATION WITH VV/VH SEPARATE
#---------------------------------------------
def viz_batch_metrics(x, y, pred_logits, thr=0.3, show_metrics=True):

    # preprocess I/O
    x = x.detach().cpu().numpy()
    if y.ndim == 4: y = y.squeeze(1)
    y = y.detach().cpu().numpy()

    # 1-channel sigmoid vs 2-class argmax
    if pred_logits.shape[1] == 1:
        probs = torch.sigmoid(pred_logits).squeeze(1).cpu().numpy() # B,H,W
        pred = (probs > thr).astype(np.uint8)
    else:
        probs = torch.softmax(pred_logits, dim=1)[:,1,:,:].cpu().numpy() # class 1 prob
        pred = torch.argmax(pred_logits, dim=1).cpu().numpy()

    b = min(3, x.shape[0])

    for i in range(b):

        vv = x[i,0]
        vh = x[i,1]
        vv_norm = (vv - vv.min())/(vv.max()-vv.min()+1e-6)
        vh_norm = (vh - vh.min())/(vh.max()-vh.min()+1e-6)

        #---------------------------------------------
        # IMAGE DISPLAY
        #---------------------------------------------
        plt.figure(figsize=(14,4))

        plt.subplot(1,5,1)
        plt.imshow(vv_norm, cmap='gray')
        plt.title('VV')
        plt.axis('off')

        plt.subplot(1,5,2)
        plt.imshow(vh_norm, cmap='gray')
        plt.title('VH')
        plt.axis('off')

        plt.subplot(1,5,3)
        plt.imshow(y[i], cmap='gray')
        plt.title("GT Mask")
        plt.axis("off")

        plt.subplot(1,5,4)
        plt.imshow(pred[i], cmap='gray')
        plt.title("Pred Mask")
        plt.axis("off")

        plt.subplot(1,5,5)
        plt.imshow((pred[i]>0)&(y[i]==0), cmap='Reds')
        plt.title("False Positives (Red)")
        plt.axis("off")

        plt.show()

        #---------------------------------------------
        # METRICS
        #---------------------------------------------
        if show_metrics:
            iou, f1 = compute_iou_f1(y[i], pred[i])
            ece = compute_ece(probs[i], y[i])

            print(f"Image {i}:")
            print(f"  IoU: {iou:.4f}")
            print(f"  F1 Score: {f1:.4f}")
            print(f"  ECE: {ece:.4f}")

            # reliability plot
            plot_reliability_diagram(probs[i], y[i])


#---------------------------------------------
# 📌 COMPUTE METRICS ON ENTIRE FOLDER
#---------------------------------------------
def evaluate_full_dataset(model, dataset, device):
    loader = DataLoader(dataset, batch_size=3, shuffle=False)

    all_iou = []
    all_f1 = []
    all_ece = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device).squeeze(1).cpu().numpy()   # (H, W)

            out = model(x)["out"]                      # (1,1,H',W')
            probs = torch.sigmoid(out).squeeze(0).squeeze(0).cpu().numpy()

            # ------------------------------
            # FIX SHAPE MISMATCH HERE
            # ------------------------------
            if probs.shape != y.shape:
                from skimage.transform import resize
                probs = resize(
                    probs,
                    y.shape,
                    order=1,
                    preserve_range=True,
                    anti_aliasing=True
                )

            pred = (probs > 0.3).astype(np.uint8)

            iou, f1 = compute_iou_f1(y, pred)
            ece = compute_ece(probs, y)

            all_iou.append(iou)
            all_f1.append(f1)
            all_ece.append(ece)

    print("\n====== OVERALL DATASET METRICS ======")
    print(f"Mean IoU: {np.mean(all_iou):.4f}")
    print(f"Mean F1:  {np.mean(all_f1):.4f}")
    print(f"Mean ECE: {np.mean(all_ece):.4f}")
    print("=====================================\n")


In [ ]:
val_items   = [all_items[102]]


#val_ds   = OilSpillDataset(val_items,   crop_size=cfg.crop_size, augment=False)
val_ds   = OilSpillDataset(val_items,   crop_size=None, augment=False)
with torch.no_grad():
    for x, y in DataLoader(val_ds, batch_size=3, shuffle=True):
        x = x.to(device); y = y.to(device)
        out = model(x)['out']
        viz_batch_metrics(x, y, out, thr=0.3, show_metrics=True)
        break


run this to evaluate full validation dataset

In [ ]:
evaluate_full_dataset(model, val_ds, device)


In [ ]:
import torch
import torch.nn as nn
from torchvision.models.segmentation import deeplabv3_resnet50

def load_deeplab_model(ckpt_path, device, in_ch=2, num_classes=2):
    # ---- 1) build SAME architecture ----
    model = deeplabv3_resnet50(weights=None, num_classes=num_classes)

    # ---- 2) patch first conv to accept 2ch ----
    old_conv = model.backbone.conv1
    if old_conv.in_channels != in_ch:
        new_conv = nn.Conv2d(
            in_ch,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=(old_conv.bias is not None)
        )
        with torch.no_grad():
            avg_w = old_conv.weight.mean(dim=1, keepdim=True)
            new_conv.weight.copy_(avg_w.repeat(1, in_ch, 1, 1))
        model.backbone.conv1 = new_conv

    # ---- 3) load checkpoint (PyTorch 2.6 fix) ----
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    # ---- 4) pick the actual state_dict robustly ----
    if isinstance(ckpt, dict):
        if "model" in ckpt:
            sd = ckpt["model"]
        elif "state_dict" in ckpt:
            sd = ckpt["state_dict"]
        elif "model_state_dict" in ckpt:
            sd = ckpt["model_state_dict"]
        else:
            sd = ckpt
    else:
        sd = ckpt

    # strip DataParallel prefix if present
    sd = {k.replace("module.", ""): v for k, v in sd.items()}

    # ---- 5) load and REPORT what happened ----
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print("✅ Loaded checkpoint.")
    print("Missing keys  :", len(missing))
    print("Unexpected keys:", len(unexpected))
    if len(missing) < 20: print("Missing sample:", missing[:10])
    if len(unexpected) < 20: print("Unexpected sample:", unexpected[:10])

    model.to(device).eval()

    # ---- 6) quick checksum so you KNOW weights changed ----
    wsum = float(model.backbone.conv1.weight.abs().mean().item())
    print("conv1 abs-mean after load:", wsum)

    return model

# ---- load ----
ckpt_path = "/content/drive/MyDrive/oil_spill_checkpoints/periments/deepLabV3/deeplabv3_resnet50_oil_fold1_best.pt"

# IMPORTANT: your fold model is almost certainly 2 classes
model = load_deeplab_model(
    ckpt_path,
    device=device,
    in_ch=2,
    num_classes=2   # <-- use 2 unless you *know* you trained different
)


In [ ]:
import copy, os
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# -----------------------------
# Viz (unchanged look, but returns probs too)
# -----------------------------
def viz_batch2(x, y, pred_logits, thr=0.3):
    x = x.detach().cpu().numpy()
    if y.ndim == 4: y = y.squeeze(1)
    y = y.detach().cpu().numpy()

    # Prob + pred
    if pred_logits.shape[1] == 1:
        prob = torch.sigmoid(pred_logits)[:,0]                     # B,H,W
        pred = (prob > thr).to(torch.uint8).cpu().numpy()          # B,H,W
        prob_np = prob.cpu().numpy()
    else:
        prob = torch.softmax(pred_logits, dim=1)[:,1]              # B,H,W (class1=oil)
        pred = (prob > thr).to(torch.uint8).cpu().numpy()          # thresholded
        prob_np = prob.cpu().numpy()

    b = min(3, x.shape[0])
    for i in range(b):
        vv = x[i,0]; vh = x[i,1]
        vv = (vv - vv.min())/(vv.max()-vv.min()+1e-6)
        vh = (vh - vh.min())/(vh.max()-vh.min()+1e-6)
        rgb = np.stack([vv, vh, 0.5*vv+0.5*vh], axis=-1)

        plt.figure(figsize=(12,3))
        plt.subplot(1,4,1); plt.imshow(rgb); plt.title('VV/VH composite'); plt.axis('off')
        plt.subplot(1,4,2); plt.imshow(y[i], cmap='gray'); plt.title('GT mask'); plt.axis('off')
        plt.subplot(1,4,3); plt.imshow(pred[i], cmap='gray'); plt.title('Pred mask'); plt.axis('off')
        plt.subplot(1,4,4); plt.imshow((pred[i]>0)&(y[i]==0), cmap='Reds'); plt.title('FP (red)'); plt.axis('off')
        plt.show()

    return pred, prob_np, y


# -----------------------------
# Metrics + reliability trend
# -----------------------------
def compute_metrics_and_reliability(prob, y_true, thr=0.5, n_bins=15, eps=1e-9):
    """
    prob: (H,W) float in [0,1]
    y_true: (H,W) uint8 {0,1}
    """
    p = prob.reshape(-1)
    y = y_true.reshape(-1).astype(np.uint8)
    pred = (p > thr).astype(np.uint8)

    TP = np.sum((pred==1) & (y==1))
    FP = np.sum((pred==1) & (y==0))
    FN = np.sum((pred==0) & (y==1))
    TN = np.sum((pred==0) & (y==0))

    iou = TP / (TP + FP + FN + eps)
    f1  = 2*TP / (2*TP + FP + FN + eps)
    far = FP / (FP + TN + eps)  # false alarm rate (false positive rate)

    # ---- ECE + reliability bins ----
    bins = np.linspace(0, 1, n_bins+1)
    bin_ids = np.digitize(p, bins) - 1

    ece = 0.0
    bin_conf, bin_acc, bin_frac = [], [], []

    for b in range(n_bins):
        m = (bin_ids == b)
        if not np.any(m):
            bin_conf.append(np.nan)
            bin_acc.append(np.nan)
            bin_frac.append(0.0)
            continue

        conf = p[m].mean()
        acc  = (pred[m] == y[m]).mean()
        frac = m.mean()

        ece += np.abs(acc - conf) * frac
        bin_conf.append(conf)
        bin_acc.append(acc)
        bin_frac.append(frac)

    metrics = dict(IoU=iou, F1=f1, FAR=far, ECE=ece,
                   TP=int(TP), FP=int(FP), FN=int(FN), TN=int(TN),
                   bin_conf=np.array(bin_conf),
                   bin_acc=np.array(bin_acc),
                   bin_frac=np.array(bin_frac))
    return metrics


def plot_reliability_trend(metrics, title="Reliability trend"):
    conf = metrics["bin_conf"]
    acc  = metrics["bin_acc"]

    # Only plot bins that exist
    m = ~np.isnan(conf) & ~np.isnan(acc)
    conf = conf[m]; acc = acc[m]

    plt.figure(figsize=(5,5))
    plt.plot([0,1],[0,1], linestyle="--")  # perfect calibration
    plt.plot(conf, acc, marker="o")
    plt.xlabel("Confidence (mean prob)")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()


# ---------------------------------------------------------
# Build a ONE-ITEM dataset using the same preprocessing
# ---------------------------------------------------------
def make_item_from_template(template_item, img_path, mask_path=None):
    it = copy.deepcopy(template_item)
    it["img"] = img_path
    if mask_path is not None:
        for k in ["mask", "msk", "label", "lbl", "gt", "seg", "y"]:
            if k in it:
                it[k] = mask_path
                break
        else:
            it["mask"] = mask_path
    return it


def infer_single_file_like_val(model, img_path, mask_path=None, thr=0.5, n_bins=15):
    template = all_items[0]
    one_item = make_item_from_template(template, img_path, mask_path)

    one_ds = OilSpillDataset([one_item], crop_size=None, augment=False)
    loader = DataLoader(one_ds, batch_size=1, shuffle=False)

    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device); y = y.to(device)
            out = model(x)["out"]

            pred, prob_np, y_np = viz_batch2(x, y, out, thr=thr)

            # ---- metrics only if GT exists ----
            if mask_path is not None:
                prob0 = prob_np[0]        # (H,W)
                gt0   = y_np[0]           # (H,W)

                metrics = compute_metrics_and_reliability(prob0, gt0, thr=thr, n_bins=n_bins)

                print("\n=== Single-file metrics ===")
                print(f"IoU: {metrics['IoU']:.4f}")
                print(f"F1 : {metrics['F1']:.4f}")
                print(f"FAR: {metrics['FAR']:.6f}")
                print(f"ECE: {metrics['ECE']:.4f}")
                print(f"TP/FP/FN/TN: {metrics['TP']}/{metrics['FP']}/{metrics['FN']}/{metrics['TN']}")

                plot_reliability_trend(metrics, title=os.path.basename(img_path))

            return out


# ---------------------------------------------------------
# Example usage
# ---------------------------------------------------------
img_path  = "/content/drive/MyDrive/Sentinel SAR datasets/02_Test_images_and_ground_truth/Images/Oil/00109.tif"
mask_path = "/content/drive/MyDrive/Sentinel SAR datasets/02_Test_images_and_ground_truth/Mask/Oil/00109_segmentation.tif"

_ = infer_single_file_like_val(model, img_path, mask_path, thr=0.2, n_bins=15)


In [ ]:
img_path  = "/content/drive/MyDrive/Sentinel SAR datasets/02_Test_images_and_ground_truth/Images/Oil/00030.tif"
mask_path = "/content/drive/MyDrive/Sentinel SAR datasets/02_Test_images_and_ground_truth/Mask/Oil/00030_segmentation.tif"

img_path  = "/content/drive/MyDrive/Sentinel SAR datasets/Oil/00030.tif"
mask_path = "/content/drive/MyDrive/Sentinel SAR datasets/Mask_oil/00030.tif"

prob_oil, preds, gt = debug_infer_one(
    model, img_path, mask_path,
    device=device, cfg=cfg, band_idxs=(1,2)
)


In [ ]:
#@title 📦 Export helpers (TorchScript / ONNX)
ckpt_path = os.path.join(cfg.out_dir, 'deeplabv3_resnet50_oil_best.pt')
bundle_dir = os.path.join(cfg.out_dir, 'bundle_deeplabv3')
os.makedirs(bundle_dir, exist_ok=True)

# Reload best
state = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict(state['state_dict'])
model.eval().to('cpu')

dummy = torch.randn(1,2,512,512)
torchscript_path = os.path.join(bundle_dir, 'deeplabv3_oil.ts')
traced = torch.jit.trace(model, dummy)
traced.save(torchscript_path)
print('Saved TorchScript:', torchscript_path)

try:
    import onnx
    onnx_path = os.path.join(bundle_dir, 'deeplabv3_oil.onnx')
    torch.onnx.export(model, dummy, onnx_path, input_names=['input'], output_names=['logits'], opset_version=17)
    print('Saved ONNX:', onnx_path)
except Exception as e:
    print('ONNX export skipped:', e)

## 🔧 Inference snippet (for your server)
Use the same normalization as the dataset above and feed a 2‑channel tensor shaped **[1,2,H,W]**. The model outputs class logits **[1,2,H,W]**; take `argmax` to get binary mask.


In [ ]:
# Example (CPU inference):
def load_deeplab_model(ckpt):
    m = deeplabv3_resnet50(weights=None, num_classes=2)
    #sd = torch.load(ckpt, map_location='cpu')['state_dict']
    #m.load_state_dict(sd);
    m.eval()
    return m

def preprocess_2ch_tiff(path):
    with rasterio.open(path) as ds:
        arr = ds.read()  # (2,H,W)
    if arr.shape[0]!=2:
        raise ValueError('Expected 2-channel tiff (VV,VH).')
    x = np.transpose(arr, (1,2,0)).astype(np.float32)
    x = np.clip(x, -50, 5)
    x = (x - (-22.5)) / 12.5
    x = np.transpose(x, (2,0,1))[None, ...]  # (1,2,H,W)
    return torch.from_numpy(x)

model = load_deeplab_model('/content/drive/MyDrive/oil_spill_checkpoints/periments/deepLabV3/deeplabv3_resnet50_oil_fold1_best.pt')
x = preprocess_2ch_tiff('/content/drive/MyDrive/Sentinel SAR datasets/02_Test_images_and_ground_truth/Images/Oil/00000.tif')
with torch.no_grad():
     logits = model(x)['out']
     pred = torch.argmax(logits, dim=1).numpy()[0].astype(np.uint8)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models.segmentation import deeplabv3_resnet50

def load_deeplab_model_2ch(ckpt_path, num_classes=2, device='cpu'):
    # 1) Build the same architecture as you used in training
    model = deeplabv3_resnet50(weights=None, num_classes=num_classes)

    # 🔑 Make sure first conv expects 2 channels (VV, VH)
    model.backbone.conv1 = nn.Conv2d(
        in_channels=2,
        out_channels=64,
        kernel_size=7,
        stride=2,
        padding=3,
        bias=False
    )

    # 2) Load the checkpoint
    ckpt = torch.load(ckpt_path, map_location=device)

    # 3) Extract the actual state_dict
    if isinstance(ckpt, dict):
        if 'state_dict' in ckpt:
            sd = ckpt['state_dict']
        elif 'model_state_dict' in ckpt:
            sd = ckpt['model_state_dict']
        elif 'model' in ckpt:
            sd = ckpt['model']
        else:
            # assume the dict itself is a state_dict
            sd = ckpt
    else:
        sd = ckpt

    # 4) Clean common prefixes like 'module.' or 'model.'
    cleaned_sd = {}
    for k, v in sd.items():
        if k.startswith('module.'):
            k = k[len('module.'):]
        if k.startswith('model.'):
            k = k[len('model.'):]
        cleaned_sd[k] = v

    # (Optional) Sanity check conv1 weight shape before loading
    print("Checkpoint conv1 weight shape:",
          cleaned_sd.get('backbone.conv1.weight', 'MISSING'))

    # 5) Load weights
    missing, unexpected = model.load_state_dict(cleaned_sd, strict=False)
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)

    model.to(device)
    model.eval()
    return model



In [ ]:
import numpy as np
import rasterio

def preprocess_2ch_tiff(path, device='cpu'):
    with rasterio.open(path) as ds:
        arr = ds.read()  # (C,H,W)

    if arr.ndim != 3 or arr.shape[0] < 2:
        raise ValueError(f"Expected at least 2 bands (VV,VH), got shape {arr.shape}")

    # Take first two bands as VV, VH
    arr = arr[:2, :, :]  # (2,H,W)

    # (H,W,2)
    x = np.transpose(arr, (1, 2, 0)).astype(np.float32)

    # Same normalization as training
    x = np.clip(x, -50, 5)
    x = (x - (-22.5)) / 12.5

    # (1,2,H,W)
    x = np.transpose(x, (2, 0, 1))[None, ...]
    t = torch.from_numpy(x).to(device)
    return t



In [ ]:
ckpt_path = '/content/drive/MyDrive/oil_spill_checkpoints/periments/deepLabV3/deeplabv3_resnet50_oil_fold1_best.pt'
img_path  = '/content/drive/MyDrive/Sentinel SAR datasets/02_Test_images_and_ground_truth/Images/Oil/00000.tif'

model = load_deeplab_model_2ch(ckpt_path)
x = preprocess_2ch_tiff(img_path)   # (1,2,H,W)

with torch.no_grad():
    out = model(x)['out']           # (1,2,H,W)
    probs = torch.softmax(out, dim=1)[0, 1]  # class 1 = oil
    pred_mask = (probs > 0.5).cpu().numpy().astype(np.uint8)
